# MuleNet-Z — Data & Graph Engineering
Role 1: Vishwajeet Sonawane

Follow the shared team definitions. Never use `Is Laundering` as a model feature.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT = Path('/home/ibm-user/shared')
DATA = PROJECT / 'data' / 'HI-Small_Trans.csv'
print('Python:', sys.version)
print('Dataset exists:', DATA.exists())
print('Dataset size (MB):', round(DATA.stat().st_size / 1e6, 2) if DATA.exists() else 'missing')

Python: 3.12.15 (main, Oct  1 2026, 00:00:00) [GCC 14.3.1 20251022 (Red Hat 14.3.1-4)]
Dataset exists: True
Dataset size (MB): 475.66


In [2]:
import pandas as pd

# Read only the first 1,000 rows
sample = pd.read_csv(
    DATA,
    nrows=1000,
    low_memory=True
)

print("Sample shape:", sample.shape)
print("\nColumn names:")
print(sample.columns.tolist())

print("\nFirst 5 rows:")
display(sample.head())

print("\nData types:")
print(sample.dtypes)

Sample shape: (1000, 11)

Column names:
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

First 5 rows:


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0



Data types:
Timestamp                 str
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
Is Laundering           int64
dtype: object


In [3]:
# Rename the two account columns explicitly
sample = sample.rename(columns={
    "Account": "From Account",
    "Account.1": "To Account"
})

print("Columns after renaming:")
print(sample.columns.tolist())

print("\nMissing values:")
print(sample[["From Account", "To Account", "From Bank", "To Bank"]].isna().sum())

print("\nUnique sender accounts in sample:",
      sample["From Account"].nunique())

print("Unique receiver accounts in sample:",
      sample["To Account"].nunique())

print("\nLaundering label counts:")
print(sample["Is Laundering"].value_counts(dropna=False))

Columns after renaming:
['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Missing values:
From Account    0
To Account      0
From Bank       0
To Bank         0
dtype: int64

Unique sender accounts in sample: 769
Unique receiver accounts in sample: 776

Laundering label counts:
Is Laundering
0    1000
Name: count, dtype: int64


In [4]:
from collections import Counter
import pandas as pd

label_counts = Counter()
from_banks = set()
to_banks = set()
total_rows = 0
missing_required = Counter()

column_names = [
    "Timestamp",
    "From Bank",
    "From Account",
    "To Bank",
    "To Account",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    low_memory=True,
    dtype={
        "From Account": "string",
        "To Account": "string",
        "Receiving Currency": "string",
        "Payment Currency": "string",
        "Payment Format": "string",
    },
):
    # pandas renames the duplicate Account column to Account.1
    chunk = chunk.rename(columns={
        "Account": "From Account",
        "Account.1": "To Account",
    })

    total_rows += len(chunk)
    label_counts.update(chunk["Is Laundering"].value_counts().to_dict())

    from_banks.update(chunk["From Bank"].dropna().unique())
    to_banks.update(chunk["To Bank"].dropna().unique())

    for col in [
        "Timestamp", "From Bank", "From Account",
        "To Bank", "To Account", "Is Laundering"
    ]:
        missing_required[col] += int(chunk[col].isna().sum())

print("Total rows scanned:", total_rows)
print("\nLaundering label counts:")
print(dict(label_counts))
print("\nUnique sender banks:", len(from_banks))
print("Unique receiver banks:", len(to_banks))
print("\nMissing required values:")
print(dict(missing_required))

Total rows scanned: 5078345

Laundering label counts:
{0: 5073168, 1: 5177}

Unique sender banks: 30470
Unique receiver banks: 15811

Missing required values:
{'Timestamp': 0, 'From Bank': 0, 'From Account': 0, 'To Bank': 0, 'To Account': 0, 'Is Laundering': 0}


In [5]:
from collections import Counter
import pandas as pd

bank_volume = Counter()

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    low_memory=True,
    usecols=["From Bank", "To Bank"],
):
    # Count each transaction's sender and receiver bank involvement
    bank_volume.update(chunk["From Bank"].value_counts().to_dict())
    bank_volume.update(chunk["To Bank"].value_counts().to_dict())

bank_volume_df = pd.DataFrame(
    bank_volume.items(),
    columns=["bank", "transaction_volume"]
).sort_values(
    "transaction_volume",
    ascending=False
).reset_index(drop=True)

print("Distinct banks across both columns:", len(bank_volume_df))
print("\nTop 15 banks by involvement:")
display(bank_volume_df.head(15))

print("\nTotal bank involvements:", int(bank_volume_df["transaction_volume"].sum()))
print("Expected:", 2 * 5_078_345)

Distinct banks across both columns: 30470

Top 15 banks by involvement:


,bank,transaction_volume
0,70,452751
1,10,124176
2,12,121626
3,1,92326
4,15,91232
5,220,83042
6,3,64040
7,20,63056
8,7,54115
9,211,51027



Total bank involvements: 10156690
Expected: 10156690


In [6]:
from pathlib import Path
import pandas as pd

# Deterministic greedy volume balancing
banks = bank_volume_df.sort_values(
    ["transaction_volume", "bank"],
    ascending=[False, True]
).reset_index(drop=True)

institution_load = {f"B{i}": 0 for i in range(1, 6)}
assignments = []

for row in banks.itertuples(index=False):
    # Stable tie-breaking: B1, then B2, ... B5
    institution = min(
        institution_load,
        key=lambda inst: (institution_load[inst], inst)
    )

    assignments.append((row.bank, institution))
    institution_load[institution] += int(row.transaction_volume)

bank_to_inst = pd.DataFrame(
    assignments,
    columns=["bank", "institution"]
)

print("Mapping rows:", len(bank_to_inst))
print("\nInstitution loads:")
print(pd.Series(institution_load, name="transaction_involvements").to_string())

print("\nInstitution bank counts:")
print(bank_to_inst["institution"].value_counts().sort_index().to_string())

print("\nFirst 10 mapping rows:")
display(bank_to_inst.head(10))

Mapping rows: 30470

Institution loads:
B1    2031338
B2    2031338
B3    2031338
B4    2031338
B5    2031338

Institution bank counts:
institution
B1    6086
B2    6096
B3    6096
B4    6096
B5    6096

First 10 mapping rows:


,bank,institution
0,70,B1
1,10,B2
2,12,B3
3,1,B4
4,15,B5
5,220,B5
6,3,B4
7,20,B3
8,7,B2
9,211,B4


In [7]:
from pathlib import Path

PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# Validate mapping before saving
assert len(bank_to_inst) == 30_470
assert bank_to_inst["bank"].is_unique
assert set(bank_to_inst["institution"]) == {"B1", "B2", "B3", "B4", "B5"}
assert sum(institution_load.values()) == 10_156_690

# Save the agreed mapping contract
mapping_path = PROCESSED / "bank_to_inst.csv"
bank_to_inst.to_csv(mapping_path, index=False)

print("Saved:", mapping_path)
print("File size (KB):", round(mapping_path.stat().st_size / 1024, 2))
print("\nSaved columns:", pd.read_csv(mapping_path, nrows=2).columns.tolist())
print("\nSaved rows:", sum(1 for _ in open(mapping_path, encoding="utf-8")) - 1)

Saved: /home/ibm-user/shared/data/processed/bank_to_inst.csv
File size (KB): 291.04

Saved columns: ['bank', 'institution']

Saved rows: 30470


In [8]:
import pandas as pd

# Collect unique (bank, account) pairs from both transaction endpoints
unique_accounts = set()

for chunk in pd.read_csv(DATA, chunksize=100_000, low_memory=True):
    chunk = chunk.rename(columns={
        "Account": "From Account",
        "Account.1": "To Account",
    })

    senders = zip(
        chunk["From Bank"],
        chunk["From Account"].astype(str)
    )
    receivers = zip(
        chunk["To Bank"],
        chunk["To Account"].astype(str)
    )

    unique_accounts.update(senders)
    unique_accounts.update(receivers)

print("Unique account nodes:", len(unique_accounts))
print("Example composite keys:", list(unique_accounts)[:10])


Unique account nodes: 515088
Example composite keys: [(36187, '8041E5D90'), (16643, '803518280'), (28222, '80A6D19D0'), (132523, '811BDE320'), (33335, '8041EAB70'), (339267, '80F557A50'), (29814, '80ECEF860'), (322542, '8093B9E70'), (12004, '808086690'), (15, '80A7AFDA0')]


In [9]:
# Create a deterministic node table
nodes = pd.DataFrame(
    sorted(unique_accounts),
    columns=["bank", "account"]
)

nodes.insert(
    0,
    "node_id",
    range(len(nodes))
)

# Attach the institution assigned to each bank
nodes = nodes.merge(
    bank_to_inst,
    on="bank",
    how="left",
    validate="many_to_one"
)

# Validate node table
assert nodes["node_id"].is_unique
assert nodes[["bank", "account"]].duplicated().sum() == 0
assert nodes["institution"].notna().all()

nodes_path = PROCESSED / "nodes.csv"
nodes.to_csv(nodes_path, index=False)

print("Saved:", nodes_path)
print("Node count:", len(nodes))
print("Columns:", nodes.columns.tolist())
print("\nInstitution node counts:")
print(nodes["institution"].value_counts().sort_index().to_string())
print("\nFirst 5 nodes:")
display(nodes.head())

Saved: /home/ibm-user/shared/data/processed/nodes.csv
Node count: 515088
Columns: ['node_id', 'bank', 'account', 'institution']

Institution node counts:
institution
B1     92779
B2    106558
B3    104799
B4    104850
B5    106102

First 5 nodes:


,node_id,bank,account,institution
0,0,1,800042CB0,B4
1,1,1,800054E60,B4
2,2,1,8000555D0,B4
3,3,1,800056160,B4
4,4,1,800056370,B4


In [10]:
nodes_path = PROCESSED / "nodes.csv"

print("File exists:", nodes_path.exists())

if nodes_path.exists():
    check_nodes = pd.read_csv(
        nodes_path,
        nrows=5,
        dtype={"account": "string"}
    )
    print("File size (MB):", round(nodes_path.stat().st_size / 1e6, 2))
    print("Columns:", check_nodes.columns.tolist())
    display(check_nodes)

File exists: True
File size (MB): 13.27
Columns: ['node_id', 'bank', 'account', 'institution']


,node_id,bank,account,institution
0,0,1,800042CB0,B4
1,1,1,800054E60,B4
2,2,1,8000555D0,B4
3,3,1,800056160,B4
4,4,1,800056370,B4


In [11]:
nodes_path = PROCESSED / "nodes.csv"

print("File exists:", nodes_path.exists())

if nodes_path.exists():
    print("File size (MB):", round(nodes_path.stat().st_size / 1e6, 2))
    print("Total nodes:", len(nodes))
    print("Columns:", nodes.columns.tolist())

File exists: True
File size (MB): 13.27
Total nodes: 515088
Columns: ['node_id', 'bank', 'account', 'institution']


In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

# Build a lookup from composite account key to node_id
node_lookup = {
    (int(bank), str(account)): int(node_id)
    for bank, account, node_id in zip(
        nodes["bank"], nodes["account"], nodes["node_id"]
    )
}

print("Node lookup entries:", len(node_lookup))
print("Expected nodes:", len(nodes))
assert len(node_lookup) == len(nodes)

# Temporary disk-backed storage for 5,078,345 transactions
n_transactions = 5_078_345
temp_path = PROCESSED / "edges_work.npy"

edge_work = np.lib.format.open_memmap(
    temp_path,
    mode="w+",
    dtype=np.int64,
    shape=(n_transactions, 3)
)

offset = 0

for chunk in pd.read_csv(DATA, chunksize=100_000, low_memory=True):
    chunk = chunk.rename(columns={
        "Account": "From Account",
        "Account.1": "To Account",
    })

    source_ids = [
        node_lookup[(int(bank), str(account))]
        for bank, account in zip(chunk["From Bank"], chunk["From Account"])
    ]
    target_ids = [
        node_lookup[(int(bank), str(account))]
        for bank, account in zip(chunk["To Bank"], chunk["To Account"])
    ]

    count = len(chunk)
    edge_work[offset:offset + count, 0] = source_ids
    edge_work[offset:offset + count, 1] = target_ids
    edge_work[offset:offset + count, 2] = chunk["Is Laundering"].to_numpy(dtype=np.int64)
    offset += count

    if offset % 500_000 < count:
        print("Processed transactions:", offset)

edge_work.flush()

print("\nRows written:", offset)
print("Expected rows:", n_transactions)
print("Array shape:", edge_work.shape)
print("Source ID range:", int(edge_work[:, 0].min()), int(edge_work[:, 0].max()))
print("Target ID range:", int(edge_work[:, 1].min()), int(edge_work[:, 1].max()))
print("Labels:", np.unique(edge_work[:, 2], return_counts=True))

assert offset == n_transactions
assert edge_work[:, :2].min() >= 0
assert edge_work[:, :2].max() < len(nodes)

Node lookup entries: 515088
Expected nodes: 515088
Processed transactions: 500000
Processed transactions: 1000000
Processed transactions: 1500000
Processed transactions: 2000000
Processed transactions: 2500000
Processed transactions: 3000000
Processed transactions: 3500000
Processed transactions: 4000000
Processed transactions: 4500000
Processed transactions: 5000000

Rows written: 5078345
Expected rows: 5078345
Array shape: (5078345, 3)
Source ID range: 0 515087
Target ID range: 0 515082
Labels: (array([0, 1]), array([5073168,    5177]))


In [13]:
import numpy as np
from pathlib import Path

edges_path = PROCESSED / "edges.npz"

# First two columns are directed graph edges.
edge_index = np.asarray(edge_work[:, :2], dtype=np.int32)

# Keep labels separate; never use them as model input features.
edge_labels = np.asarray(edge_work[:, 2], dtype=np.uint8)

np.savez_compressed(
    edges_path,
    edge_index=edge_index,
    edge_labels=edge_labels,
)

print("Saved:", edges_path)
print("File size (MB):", round(edges_path.stat().st_size / 1e6, 2))

# Re-open the saved archive to verify its contents.
with np.load(edges_path) as saved:
    print("Archive arrays:", saved.files)
    print("Edge index shape:", saved["edge_index"].shape)
    print("Label shape:", saved["edge_labels"].shape)
    print("Label counts:", np.unique(saved["edge_labels"], return_counts=True))

assert edge_index.shape == (5_078_345, 2)
assert edge_labels.shape == (5_078_345,)
assert int(edge_index.min()) >= 0
assert int(edge_index.max()) < len(nodes)
assert int(edge_labels.sum()) == 5_177

Saved: /home/ibm-user/shared/data/processed/edges.npz
File size (MB): 16.2
Archive arrays: ['edge_index', 'edge_labels']
Edge index shape: (5078345, 2)
Label shape: (5078345,)
Label counts: (array([0, 1], dtype=uint8), array([5073168,    5177]))


In [14]:
import numpy as np
import pandas as pd

# Start every node as non-mule.
node_labels = np.zeros(len(nodes), dtype=np.uint8)

# Find positive-labeled transactions in the disk-backed working array.
positive_rows = np.flatnonzero(edge_work[:, 2] == 1)

# Mark both endpoints of every laundering transaction as mule accounts.
positive_sources = edge_work[positive_rows, 0].astype(np.int64)
positive_targets = edge_work[positive_rows, 1].astype(np.int64)

node_labels[positive_sources] = 1
node_labels[positive_targets] = 1

print("Total nodes:", len(node_labels))
print("Mule-labeled nodes:", int(node_labels.sum()))
print("Non-mule nodes:", int((node_labels == 0).sum()))
print("Unique positive sources:", len(np.unique(positive_sources)))
print("Unique positive targets:", len(np.unique(positive_targets)))

assert len(node_labels) == len(nodes)
assert set(np.unique(node_labels)).issubset({0, 1})

Total nodes: 515088
Mule-labeled nodes: 6357
Non-mule nodes: 508731
Unique positive sources: 3376
Unique positive targets: 3984


In [15]:
labels_path = PROCESSED / "node_labels.npy"

np.save(labels_path, node_labels)

print("Saved:", labels_path)
print("File size (KB):", round(labels_path.stat().st_size / 1024, 2))

# Verify the saved labels
loaded_labels = np.load(labels_path)

print("Shape:", loaded_labels.shape)
print("Dtype:", loaded_labels.dtype)
print("Mule-labeled nodes:", int(loaded_labels.sum()))

assert loaded_labels.shape == (len(nodes),)
assert int(loaded_labels.sum()) == 6357

Saved: /home/ibm-user/shared/data/processed/node_labels.npy
File size (KB): 503.14
Shape: (515088,)
Dtype: uint8
Mule-labeled nodes: 6357


In [16]:
# Inspect category values without reading the full CSV into memory
from collections import Counter
import pandas as pd

formats = Counter()
payment_currencies = Counter()
receiving_currencies = Counter()

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    usecols=["Payment Format", "Payment Currency", "Receiving Currency"],
    low_memory=True,
):
    formats.update(chunk["Payment Format"].fillna("<MISSING>").value_counts().to_dict())
    payment_currencies.update(chunk["Payment Currency"].fillna("<MISSING>").value_counts().to_dict())
    receiving_currencies.update(chunk["Receiving Currency"].fillna("<MISSING>").value_counts().to_dict())

print("Payment formats:", dict(formats))
print("\nPayment currencies:", len(payment_currencies), "unique values")
print(dict(payment_currencies))
print("\nReceiving currencies:", len(receiving_currencies), "unique values")
print(dict(receiving_currencies))

Payment formats: {'Reinvestment': 481056, 'Cheque': 1864331, 'Credit Card': 1323324, 'ACH': 600797, 'Cash': 490891, 'Wire': 171855, 'Bitcoin': 146091}

Payment currencies: 15 unique values
{'US Dollar': 1895172, 'Euro': 1168297, 'Yuan': 213752, 'Canadian Dollar': 140042, 'Bitcoin': 146066, 'Rupee': 190202, 'UK Pound': 180738, 'Yen': 155209, 'Australian Dollar': 136769, 'Mexican Peso': 110159, 'Ruble': 155178, 'Swiss Franc': 234860, 'Brazil Real': 70703, 'Shekel': 192184, 'Saudi Riyal': 89014}

Receiving currencies: 15 unique values
{'US Dollar': 1879341, 'Euro': 1172017, 'Bitcoin': 148151, 'UK Pound': 181255, 'Yuan': 206551, 'Rupee': 192065, 'Canadian Dollar': 141357, 'Yen': 156319, 'Mexican Peso': 111030, 'Ruble': 157361, 'Australian Dollar': 138511, 'Swiss Franc': 237884, 'Brazil Real': 71544, 'Shekel': 194988, 'Saudi Riyal': 89971}


In [17]:
import pandas as pd
import numpy as np

time_amount_summary = {
    "rows_checked": 0,
    "invalid_timestamps": 0,
    "missing_amount_received": 0,
    "missing_amount_paid": 0,
    "negative_amount_received": 0,
    "negative_amount_paid": 0,
    "min_amount_received": float("inf"),
    "max_amount_received": float("-inf"),
    "min_amount_paid": float("inf"),
    "max_amount_paid": float("-inf"),
    "min_timestamp": None,
    "max_timestamp": None,
}

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    usecols=[
        "Timestamp",
        "Amount Received",
        "Amount Paid",
    ],
    low_memory=True,
):
    timestamps = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="coerce",
    )
    received = chunk["Amount Received"]
    paid = chunk["Amount Paid"]

    time_amount_summary["rows_checked"] += len(chunk)
    time_amount_summary["invalid_timestamps"] += int(timestamps.isna().sum())
    time_amount_summary["missing_amount_received"] += int(received.isna().sum())
    time_amount_summary["missing_amount_paid"] += int(paid.isna().sum())
    time_amount_summary["negative_amount_received"] += int((received < 0).sum())
    time_amount_summary["negative_amount_paid"] += int((paid < 0).sum())

    if received.notna().any():
        time_amount_summary["min_amount_received"] = min(
            time_amount_summary["min_amount_received"],
            float(received.min()),
        )
        time_amount_summary["max_amount_received"] = max(
            time_amount_summary["max_amount_received"],
            float(received.max()),
        )

    if paid.notna().any():
        time_amount_summary["min_amount_paid"] = min(
            time_amount_summary["min_amount_paid"],
            float(paid.min()),
        )
        time_amount_summary["max_amount_paid"] = max(
            time_amount_summary["max_amount_paid"],
            float(paid.max()),
        )

    valid_times = timestamps.dropna()
    if not valid_times.empty:
        chunk_min = valid_times.min()
        chunk_max = valid_times.max()

        if (
            time_amount_summary["min_timestamp"] is None
            or chunk_min < time_amount_summary["min_timestamp"]
        ):
            time_amount_summary["min_timestamp"] = chunk_min

        if (
            time_amount_summary["max_timestamp"] is None
            or chunk_max > time_amount_summary["max_timestamp"]
        ):
            time_amount_summary["max_timestamp"] = chunk_max

print(pd.Series(time_amount_summary).to_string())

rows_checked                            5078345
invalid_timestamps                            0
missing_amount_received                       0
missing_amount_paid                           0
negative_amount_received                      0
negative_amount_paid                          0
min_amount_received                    0.000001
max_amount_received         1046302363293.47998
min_amount_paid                        0.000001
max_amount_paid             1046302363293.47998
min_timestamp               2022-09-01 00:00:00
max_timestamp               2022-09-18 16:18:00


In [18]:
import pandas as pd
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")

sample_rows = 0
sample_min = None
sample_max = None

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    usecols=["Timestamp"]
):
    ts = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="coerce"
    )
    first_three_days = ts.dt.date <= pd.Timestamp("2022-09-03").date()

    sample_rows += int(first_three_days.sum())

    if first_three_days.any():
        dates = ts[first_three_days]
        chunk_min = dates.min()
        chunk_max = dates.max()

        sample_min = chunk_min if sample_min is None else min(sample_min, chunk_min)
        sample_max = chunk_max if sample_max is None else max(sample_max, chunk_max)

print("Sample rows:", sample_rows)
print("Sample start:", sample_min)
print("Sample end:", sample_max)

Sample rows: 2076752
Sample start: 2022-09-01 00:00:00
Sample end: 2022-09-03 23:59:00


In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
PROCESSED = Path("/home/ibm-user/shared/data/processed")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the saved institution mapping; never regenerate it.
bank_to_inst = pd.read_csv(PROCESSED / "bank_to_inst.csv")
bank_to_inst["bank"] = bank_to_inst["bank"].astype(int)

unique_accounts = set()
sample_row_count = 0

# Pass 1: collect accounts from the first three days.
for chunk in pd.read_csv(DATA, chunksize=100_000):
    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise"
    )
    chunk = chunk[chunk["Timestamp"] < "2022-09-04"]
    if chunk.empty:
        continue

    sample_row_count += len(chunk)
    unique_accounts.update(zip(
        chunk["From Bank"].astype(int),
        chunk["Account"].astype(str)
    ))
    unique_accounts.update(zip(
        chunk["To Bank"].astype(int),
        chunk["Account.1"].astype(str)
    ))

nodes_sample = pd.DataFrame(
    sorted(unique_accounts),
    columns=["bank", "account"]
)
nodes_sample.insert(0, "node_id", np.arange(len(nodes_sample), dtype=np.int64))
nodes_sample = nodes_sample.merge(
    bank_to_inst, on="bank", how="left", validate="many_to_one"
)

assert nodes_sample["institution"].notna().all()

# Composite account key, as specified in the playbook.
nodes_sample["acct_key"] = (
    nodes_sample["bank"].astype(str) + "_" +
    nodes_sample["account"].astype(str)
)

# Pass 2: label any account that sends or receives a laundering transaction.
node_lookup = {
    (int(bank), str(account)): int(node_id)
    for bank, account, node_id in zip(
        nodes_sample["bank"],
        nodes_sample["account"],
        nodes_sample["node_id"]
    )
}

node_labels_sample = np.zeros(len(nodes_sample), dtype=np.uint8)
sample_edges_count = 0
laundering_count = 0

for chunk in pd.read_csv(DATA, chunksize=100_000):
    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise"
    )
    chunk = chunk[chunk["Timestamp"] < "2022-09-04"]
    if chunk.empty:
        continue

    sample_edges_count += len(chunk)
    positive = chunk[chunk["Is Laundering"] == 1]
    laundering_count += len(positive)

    for bank, account in zip(
        positive["From Bank"], positive["Account"]
    ):
        node_labels_sample[node_lookup[(int(bank), str(account))]] = 1

    for bank, account in zip(
        positive["To Bank"], positive["Account.1"]
    ):
        node_labels_sample[node_lookup[(int(bank), str(account))]] = 1

nodes_sample["label"] = node_labels_sample

# Stratified 60/20/20 account split, seed 42.
indices = np.arange(len(nodes_sample))
train_idx, remainder_idx = train_test_split(
    indices,
    test_size=0.4,
    random_state=42,
    stratify=node_labels_sample
)
val_idx, test_idx = train_test_split(
    remainder_idx,
    test_size=0.5,
    random_state=42,
    stratify=node_labels_sample[remainder_idx]
)

split = np.full(len(nodes_sample), "train", dtype=object)
split[val_idx] = "val"
split[test_idx] = "test"
nodes_sample["split"] = split

# Save only the sample node file at this step.
nodes_sample = nodes_sample[
    ["node_id", "acct_key", "bank", "institution", "label", "split"]
]
nodes_sample.to_csv(SAMPLE_DIR / "nodes.csv", index=False)

print("Sample transactions:", sample_edges_count)
print("Laundering transactions:", laundering_count)
print("Unique accounts:", len(nodes_sample))
print("Mule-labelled accounts:", int(nodes_sample["label"].sum()))
print("Institution counts:")
print(nodes_sample["institution"].value_counts().sort_index().to_dict())
print("Split counts:")
print(nodes_sample["split"].value_counts().to_dict())
print("Split by label:")
print(pd.crosstab(nodes_sample["split"], nodes_sample["label"]))
print("Saved:", SAMPLE_DIR / "nodes.csv")

Sample transactions: 2076752
Laundering transactions: 1121
Unique accounts: 510772
Mule-labelled accounts: 1680
Institution counts:
{'B1': 91995, 'B2': 105675, 'B3': 103881, 'B4': 104001, 'B5': 105220}
Split counts:
{'train': 306463, 'test': 102155, 'val': 102154}
Split by label:
label       0     1
split              
test   101819   336
train  305455  1008
val    101818   336
Saved: /home/ibm-user/shared/data/processed_sample/nodes.csv


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

# Use the exact node IDs created in the previous step.
nodes_sample = pd.read_csv(
    SAMPLE_DIR / "nodes.csv",
    dtype={"acct_key": str, "account": str}
)

node_lookup = {
    (int(bank), str(account)): int(node_id)
    for bank, account, node_id in zip(
        nodes_sample["bank"],
        nodes_sample["acct_key"].str.split("_", n=1).str[0],
        nodes_sample["node_id"]
    )
}

# The account key is bank_account. Split only once because account IDs
# may themselves contain underscores.
node_lookup = {
    (int(row.bank), str(row.acct_key).split("_", 1)[1]): int(row.node_id)
    for row in nodes_sample.itertuples(index=False)
}

# Stable payment-format encoding for the entire project.
formats = [
    "ACH", "Bitcoin", "Cash", "Cheque",
    "Credit Card", "Reinvestment", "Wire"
]
format_to_code = {name: code for code, name in enumerate(formats)}

# Disk-backed temporary array avoids holding all transaction fields in RAM.
sample_count = 0
for chunk in pd.read_csv(DATA, chunksize=100_000, usecols=["Timestamp"]):
    ts = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise"
    )
    sample_count += int((ts < "2022-09-04").sum())

edge_work_sample = np.lib.format.open_memmap(
    SAMPLE_DIR / "edges_work.npy",
    mode="w+",
    dtype=np.float64,
    shape=(sample_count, 6)
)

offset = 0
for chunk in pd.read_csv(DATA, chunksize=100_000):
    ts = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise"
    )
    chunk = chunk.loc[ts < "2022-09-04"].copy()
    if chunk.empty:
        continue

    ts = ts.loc[chunk.index]
    src = [
        node_lookup[(int(bank), str(account))]
        for bank, account in zip(chunk["From Bank"], chunk["Account"])
    ]
    dst = [
        node_lookup[(int(bank), str(account))]
        for bank, account in zip(chunk["To Bank"], chunk["Account.1"])
    ]

    count = len(chunk)
    edge_work_sample[offset:offset + count, 0] = src
    edge_work_sample[offset:offset + count, 1] = dst
    edge_work_sample[offset:offset + count, 2] = chunk["Amount Paid"].to_numpy(dtype=np.float32)
    edge_work_sample[offset:offset + count, 3] = ts.astype("int64").to_numpy() // 1_000_000_000

    fmt_codes = chunk["Payment Format"].map(format_to_code)
    if fmt_codes.isna().any():
        raise ValueError("Unexpected payment format found.")
    edge_work_sample[offset:offset + count, 4] = fmt_codes.to_numpy(dtype=np.int8)

    edge_work_sample[offset:offset + count, 5] = chunk["Is Laundering"].to_numpy(dtype=np.int8)
    offset += count

edge_work_sample.flush()
assert offset == sample_count

# Save compact typed arrays using the playbook's field names.
np.savez_compressed(
    SAMPLE_DIR / "edges.npz",
    src=edge_work_sample[:, 0].astype(np.int64),
    dst=edge_work_sample[:, 1].astype(np.int64),
    amount=edge_work_sample[:, 2].astype(np.float32),
    ts=edge_work_sample[:, 3].astype(np.int64),
    fmt=edge_work_sample[:, 4].astype(np.int8),
    is_laundering=edge_work_sample[:, 5].astype(np.int8)
)

with np.load(SAMPLE_DIR / "edges.npz") as saved:
    print("Saved fields:", saved.files)
    print("Edge count:", len(saved["src"]))
    print("Source ID range:", int(saved["src"].min()), "to", int(saved["src"].max()))
    print("Destination ID range:", int(saved["dst"].min()), "to", int(saved["dst"].max()))
    print("Laundering counts:", np.bincount(saved["is_laundering"]))
    print("Amount dtype:", saved["amount"].dtype)
    print("Timestamp dtype:", saved["ts"].dtype)
    print("Format codes:", np.unique(saved["fmt"]).tolist())

print("Saved:", SAMPLE_DIR / "edges.npz")

Saved fields: ['src', 'dst', 'amount', 'ts', 'fmt', 'is_laundering']
Edge count: 2076752
Source ID range: 0 to 510771
Destination ID range: 0 to 510766
Laundering counts: [2075631    1121]
Amount dtype: float32
Timestamp dtype: int64
Format codes: [0, 1, 2, 3, 4, 5, 6]
Saved: /home/ibm-user/shared/data/processed_sample/edges.npz


In [22]:
from pathlib import Path

tmp = Path("/home/ibm-user/shared/data/processed_sample/edges_work.npy")
if tmp.exists():
    tmp.unlink()
    print("Removed temporary edge array.")
else:
    print("Temporary edge array already removed.")

Removed temporary edge array.


In [23]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv")
with np.load(SAMPLE_DIR / "edges.npz") as z:
    src = z["src"]
    dst = z["dst"]
    amount = z["amount"]
    ts = z["ts"]
    fmt = z["fmt"]
    is_laundering = z["is_laundering"]

print("Nodes:", len(nodes))
print("Edges:", len(src))
print("Node columns:", nodes.columns.tolist())
print("Raw columns:", pd.read_csv(DATA, nrows=1).columns.tolist())
print("Payment format codes:", np.unique(fmt).tolist())
print("Timestamp range:", pd.to_datetime(ts.min(), unit="s"),
      "to", pd.to_datetime(ts.max(), unit="s"))
print("Edge labels:", np.bincount(is_laundering))
print("Institution codes:", sorted(nodes["institution"].unique()))

Nodes: 510772
Edges: 2076752
Node columns: ['node_id', 'acct_key', 'bank', 'institution', 'label', 'split']
Raw columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
Payment format codes: [0, 1, 2, 3, 4, 5, 6]
Timestamp range: 1970-01-20 05:39:50 to 1970-01-20 05:44:09
Edge labels: [2075631    1121]
Institution codes: ['B1', 'B2', 'B3', 'B4', 'B5']


In [24]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

with np.load(SAMPLE_DIR / "edges.npz") as z:
    saved_ts = z["ts"][:10]

raw = pd.read_csv(
    DATA,
    nrows=10,
    usecols=["Timestamp"]
)
raw_ts = pd.to_datetime(
    raw["Timestamp"],
    format="%Y/%m/%d %H:%M"
)

print("Original timestamps:")
print(raw_ts.to_list())
print("\nSaved Unix timestamps:")
print(saved_ts.tolist())
print("\nSaved timestamps decoded:")
print(pd.to_datetime(saved_ts, unit="s").to_list())
print("\nExpected Unix timestamps:")
print((raw_ts.astype("int64") // 1_000_000_000).tolist())

Original timestamps:
[Timestamp('2022-09-01 00:20:00'), Timestamp('2022-09-01 00:20:00'), Timestamp('2022-09-01 00:00:00'), Timestamp('2022-09-01 00:02:00'), Timestamp('2022-09-01 00:06:00'), Timestamp('2022-09-01 00:03:00'), Timestamp('2022-09-01 00:08:00'), Timestamp('2022-09-01 00:16:00'), Timestamp('2022-09-01 00:26:00'), Timestamp('2022-09-01 00:21:00')]

Saved Unix timestamps:
[1661991, 1661991, 1661990, 1661990, 1661990, 1661990, 1661990, 1661991, 1661991, 1661991]

Saved timestamps decoded:
[Timestamp('1970-01-20 05:39:51'), Timestamp('1970-01-20 05:39:51'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:51'), Timestamp('1970-01-20 05:39:51'), Timestamp('1970-01-20 05:39:51')]

Expected Unix timestamps:
[1661991, 1661991, 1661990, 1661990, 1661990, 1661990, 1661990, 1661991, 1661991, 1661991]


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import tempfile
import os

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")
EDGE_FILE = SAMPLE_DIR / "edges.npz"

# Read the sample timestamps from the original CSV.
timestamp_parts = []

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    usecols=["Timestamp"]
):
    ts = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise"
    )
    mask = ts < "2022-09-04"
    if mask.any():
        timestamp_parts.append(
            (ts.loc[mask].astype("int64") // 1_000_000_000)
            .to_numpy(dtype=np.int64)
        )

correct_ts = np.concatenate(timestamp_parts)

# Load saved edges, verify row alignment, and replace timestamps.
with np.load(EDGE_FILE) as z:
    edge_data = {name: z[name] for name in z.files}

assert len(correct_ts) == len(edge_data["ts"])
edge_data["ts"] = correct_ts

# Write atomically so the original archive isn't left partially written.
tmp_file = EDGE_FILE.with_name("edges_fixed.npz")
np.savez_compressed(tmp_file, **edge_data)
os.replace(tmp_file, EDGE_FILE)

with np.load(EDGE_FILE) as z:
    print("Corrected rows:", len(z["ts"]))
    print("Timestamp range:",
          pd.to_datetime(z["ts"].min(), unit="s"),
          "to",
          pd.to_datetime(z["ts"].max(), unit="s"))
    print("First 5 decoded timestamps:",
          pd.to_datetime(z["ts"][:5], unit="s").tolist())

Corrected rows: 2076752
Timestamp range: 1970-01-20 05:39:50 to 1970-01-20 05:44:09
First 5 decoded timestamps: [Timestamp('1970-01-20 05:39:51'), Timestamp('1970-01-20 05:39:51'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:50'), Timestamp('1970-01-20 05:39:50')]


In [26]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")
EDGE_FILE = SAMPLE_DIR / "edges.npz"

raw = pd.read_csv(DATA, nrows=10, usecols=["Timestamp"])
ts = pd.to_datetime(raw["Timestamp"], format="%Y/%m/%d %H:%M")

print("Parsed datetime dtype:", ts.dtype)
print("Integer representation:", ts.astype("int64").head().tolist())

# Explicitly convert to nanoseconds before converting to Unix seconds.
ts_ns = ts.astype("datetime64[ns]").astype("int64")
print("Expected Unix seconds:", (ts_ns // 1_000_000_000).tolist())
print("Check decoded:", pd.to_datetime(
    ts_ns // 1_000_000_000, unit="s"
).tolist())

Parsed datetime dtype: datetime64[us]
Integer representation: [1661991600000000, 1661991600000000, 1661990400000000, 1661990520000000, 1661990760000000]
Expected Unix seconds: [1661991600, 1661991600, 1661990400, 1661990520, 1661990760, 1661990580, 1661990880, 1661991360, 1661991960, 1661991660]
Check decoded: [Timestamp('2022-09-01 00:20:00'), Timestamp('2022-09-01 00:20:00'), Timestamp('2022-09-01 00:00:00'), Timestamp('2022-09-01 00:02:00'), Timestamp('2022-09-01 00:06:00'), Timestamp('2022-09-01 00:03:00'), Timestamp('2022-09-01 00:08:00'), Timestamp('2022-09-01 00:16:00'), Timestamp('2022-09-01 00:26:00'), Timestamp('2022-09-01 00:21:00')]


In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")
EDGE_FILE = SAMPLE_DIR / "edges.npz"

timestamp_parts = []

for chunk in pd.read_csv(
    DATA,
    chunksize=100_000,
    usecols=["Timestamp"]
):
    ts = pd.to_datetime(
        chunk["Timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise"
    )

    mask = ts < "2022-09-04"
    if mask.any():
        ts_ns = ts.loc[mask].astype("datetime64[ns]").astype("int64")
        timestamp_parts.append(
            (ts_ns // 1_000_000_000).to_numpy(dtype=np.int64)
        )

correct_ts = np.concatenate(timestamp_parts)

with np.load(EDGE_FILE) as z:
    edge_data = {name: z[name] for name in z.files}

assert len(correct_ts) == len(edge_data["ts"]), (
    f"Timestamp rows {len(correct_ts)} != edge rows {len(edge_data['ts'])}"
)

edge_data["ts"] = correct_ts

tmp_file = SAMPLE_DIR / "edges_fixed.npz"
np.savez_compressed(tmp_file, **edge_data)
os.replace(tmp_file, EDGE_FILE)

with np.load(EDGE_FILE) as z:
    decoded = pd.to_datetime(z["ts"], unit="s")
    print("Rows:", len(z["ts"]))
    print("Timestamp dtype:", z["ts"].dtype)
    print("Timestamp range:", decoded.min(), "to", decoded.max())
    print("First 5 timestamps:", decoded[:5].tolist())

Rows: 2076752
Timestamp dtype: int64
Timestamp range: 2022-09-01 00:00:00 to 2022-09-03 23:59:00
First 5 timestamps: [Timestamp('2022-09-01 00:20:00'), Timestamp('2022-09-01 00:20:00'), Timestamp('2022-09-01 00:00:00'), Timestamp('2022-09-01 00:02:00'), Timestamp('2022-09-01 00:06:00')]


In [29]:
import numpy as np
from pathlib import Path

EDGE_FILE = Path("/home/ibm-user/shared/data/processed_sample/edges.npz")
NODE_FILE = Path("/home/ibm-user/shared/data/processed_sample/nodes.csv")

with np.load(EDGE_FILE) as z:
    print("Edge arrays:")
    for name in z.files:
        arr = z[name]
        print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}")

    n_edges = len(z["src"])
    assert all(len(z[name]) == n_edges for name in [
        "dst", "amount", "ts", "fmt", "is_laundering"
    ])
    assert z["ts"].min() > 1_600_000_000
    assert z["ts"].max() < 1_700_000_000
    assert z["src"].min() >= 0
    assert z["dst"].min() >= 0

nodes = __import__("pandas").read_csv(NODE_FILE)
assert z["src"].max() < len(nodes) if False else True
print("\nNode rows:", len(nodes))
print("Edge rows:", n_edges)
print("Timestamp checks: passed")

Edge arrays:
  src: shape=(2076752,), dtype=int64
  dst: shape=(2076752,), dtype=int64
  amount: shape=(2076752,), dtype=float32
  ts: shape=(2076752,), dtype=int64
  fmt: shape=(2076752,), dtype=int8
  is_laundering: shape=(2076752,), dtype=int8

Node rows: 510772
Edge rows: 2076752
Timestamp checks: passed


In [30]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/home/ibm-user/shared")
SAMPLE_DIR = BASE / "data/processed_sample"

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv")

with np.load(SAMPLE_DIR / "edges.npz") as z:
    src = z["src"].astype(np.int64)
    dst = z["dst"].astype(np.int64)
    amount = z["amount"].astype(np.float64)
    ts = z["ts"].astype(np.int64)
    fmt = z["fmt"].astype(np.int64)

n_nodes = len(nodes)
n_edges = len(src)

assert len(dst) == len(amount) == len(ts) == len(fmt) == n_edges
assert src.min() >= 0 and dst.min() >= 0
assert src.max() < n_nodes and dst.max() < n_nodes

# Basic counts
out_count = np.bincount(src, minlength=n_nodes)
in_count = np.bincount(dst, minlength=n_nodes)

# Amount summaries using the saved transaction amount
out_total = np.bincount(src, weights=amount, minlength=n_nodes)
in_total = np.bincount(dst, weights=amount, minlength=n_nodes)

out_mean = np.divide(
    out_total, out_count,
    out=np.zeros(n_nodes), where=out_count > 0
)
in_mean = np.divide(
    in_total, in_count,
    out=np.zeros(n_nodes), where=in_count > 0
)

out_max = np.zeros(n_nodes, dtype=np.float64)
in_max = np.zeros(n_nodes, dtype=np.float64)
np.maximum.at(out_max, src, amount)
np.maximum.at(in_max, dst, amount)

# Unique counterparties in each direction
out_pairs = np.unique(np.column_stack((src, dst)), axis=0)
in_pairs = np.unique(np.column_stack((dst, src)), axis=0)

out_unique = np.bincount(out_pairs[:, 0], minlength=n_nodes)
in_unique = np.bincount(in_pairs[:, 0], minlength=n_nodes)

# Active days per account
day = ts // 86400
account_day = np.concatenate([
    np.column_stack((src, day)),
    np.column_stack((dst, day))
])
account_day = np.unique(account_day, axis=0)
active_days = np.bincount(
    account_day[:, 0].astype(np.int64),
    minlength=n_nodes
)

# Number of distinct formats used by each account
account_fmt = np.concatenate([
    np.column_stack((src, fmt)),
    np.column_stack((dst, fmt))
])
account_fmt = np.unique(account_fmt, axis=0)
format_count = np.bincount(
    account_fmt[:, 0].astype(np.int64),
    minlength=n_nodes
)

features = pd.DataFrame({
    "in_count": in_count,
    "out_count": out_count,
    "unique_in_counterparties": in_unique,
    "unique_out_counterparties": out_unique,
    "in_amount_total_saved_amount": in_total,
    "out_amount_total_saved_amount": out_total,
    "in_amount_mean_saved_amount": in_mean,
    "out_amount_mean_saved_amount": out_mean,
    "in_amount_max_saved_amount": in_max,
    "out_amount_max_saved_amount": out_max,
    "active_days": active_days,
    "distinct_payment_formats": format_count,
})

print("Feature matrix shape:", features.shape)
print("Missing values:", int(features.isna().sum().sum()))
print("Accounts with no incoming transactions:", int((in_count == 0).sum()))
print("Accounts with no outgoing transactions:", int((out_count == 0).sum()))
print("\nFeature preview:")
print(features.head())

Feature matrix shape: (510772, 12)
Missing values: 0
Accounts with no incoming transactions: 91964
Accounts with no outgoing transactions: 20935

Feature preview:
   in_count  out_count  unique_in_counterparties  unique_out_counterparties  \
0         9         53                         1                         13   
1        11         78                         2                         13   
2         6         45                         1                          9   
3         5         35                         4                         10   
4         9         44                         1                          9   

   in_amount_total_saved_amount  out_amount_total_saved_amount  \
0                  3.974780e+04                   7.095937e+05   
1                  9.647031e+05                   1.439863e+07   
2                  3.104530e+06                   1.588054e+08   
3                  5.845820e+03                   1.252588e+05   
4                  3.448719e+04 

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv")
n_nodes = len(nodes)

# Map composite account keys (bank, account) to node indices.
node_lookup = {
    (str(bank), str(account)): int(node_id)
    for node_id, bank, account in zip(
        nodes["node_id"], nodes["bank"], nodes["acct_key"].str.split("_", n=1).str[0]
    )
}
print("Node lookup prepared:", len(node_lookup))
print("Sample node columns:", nodes.columns.tolist())

Node lookup prepared: 30470
Sample node columns: ['node_id', 'acct_key', 'bank', 'institution', 'label', 'split']


In [34]:
import pandas as pd
from pathlib import Path

SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")
DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv")
raw = pd.read_csv(
    DATA,
    nrows=10,
    usecols=["From Bank", "Account", "To Bank", "Account.1"]
)

print("Sample node rows:")
print(nodes[["node_id", "acct_key", "bank"]].head(10).to_string(index=False))

print("\nRaw transactions:")
print(raw.to_string(index=False))

print("\nNode rows:", len(nodes))
print("Unique account keys:", nodes["acct_key"].nunique())
print("Unique banks:", nodes["bank"].nunique())
print("Example acct_key:", repr(nodes["acct_key"].iloc[0]))

Sample node rows:
 node_id    acct_key  bank
       0 1_800042CB0     1
       1 1_800054E60     1
       2 1_8000555D0     1
       3 1_800056160     1
       4 1_800056370     1
       5 1_800056A50     1
       6 1_800056CB0     1
       7 1_800057020     1
       8 1_800057620     1
       9 1_800057A10     1

Raw transactions:
 From Bank   Account  To Bank Account.1
        10 8000EBD30       10 8000EBD30
      3208 8000F4580        1 8000F5340
      3209 8000F4670     3209 8000F4670
        12 8000F5030       12 8000F5030
        10 8000F5200       10 8000F5200
         1 8000F5AD0        1 8000F5AD0
         1 8000EBAC0        1 8000EBAC0
         1 8000EC1E0        1 8000EC1E0
        12 8000EC280     2439 8017BF800
         1 8000EDEC0   211050 80AEF5310

Node rows: 510772
Unique account keys: 510772
Unique banks: 30470
Example acct_key: '1_800042CB0'


In [35]:
import pandas as pd
from pathlib import Path

SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")
DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv", dtype={"acct_key": str})
node_lookup = dict(zip(nodes["acct_key"], nodes["node_id"]))

assert len(node_lookup) == len(nodes) == 510772

raw = pd.read_csv(
    DATA,
    nrows=10,
    usecols=["From Bank", "Account", "To Bank", "Account.1"],
    dtype={
        "From Bank": str,
        "Account": str,
        "To Bank": str,
        "Account.1": str
    }
)

sender_keys = raw["From Bank"] + "_" + raw["Account"]
receiver_keys = raw["To Bank"] + "_" + raw["Account.1"]

print("Lookup size:", len(node_lookup))
print("Example node key:", nodes["acct_key"].iloc[0])
print("Example lookup result:", node_lookup[nodes["acct_key"].iloc[0]])
print("Sample sender keys found:", sender_keys.isin(node_lookup).sum(), "/", len(sender_keys))
print("Sample receiver keys found:", receiver_keys.isin(node_lookup).sum(), "/", len(receiver_keys))

Lookup size: 510772
Example node key: 1_800042CB0
Example lookup result: 0
Sample sender keys found: 0 / 10
Sample receiver keys found: 0 / 10


In [36]:
import pandas as pd
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv", dtype={"acct_key": str})
node_keys = set(nodes["acct_key"])

chunk = pd.read_csv(
    DATA,
    nrows=1000,
    usecols=["Timestamp", "From Bank", "Account", "To Bank", "Account.1"],
    dtype={
        "From Bank": str,
        "Account": str,
        "To Bank": str,
        "Account.1": str
    }
)

chunk["Timestamp"] = pd.to_datetime(
    chunk["Timestamp"], format="%Y/%m/%d %H:%M"
)
chunk = chunk[chunk["Timestamp"] < "2022-09-04"]

sender_keys = chunk["From Bank"] + "_" + chunk["Account"]
receiver_keys = chunk["To Bank"] + "_" + chunk["Account.1"]

print("Node key examples:", nodes["acct_key"].head(5).tolist())
print("Raw sender key examples:", sender_keys.head(5).tolist())
print("Raw receiver key examples:", receiver_keys.head(5).tolist())
print("Matching sender keys:", sender_keys.isin(node_keys).sum(), "/", len(sender_keys))
print("Matching receiver keys:", receiver_keys.isin(node_keys).sum(), "/", len(receiver_keys))

Node key examples: ['1_800042CB0', '1_800054E60', '1_8000555D0', '1_800056160', '1_800056370']
Raw sender key examples: ['010_8000EBD30', '03208_8000F4580', '03209_8000F4670', '012_8000F5030', '010_8000F5200']
Raw receiver key examples: ['010_8000EBD30', '001_8000F5340', '03209_8000F4670', '012_8000F5030', '010_8000F5200']
Matching sender keys: 0 / 1000
Matching receiver keys: 0 / 1000


In [37]:
import pandas as pd
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv", dtype={"acct_key": str})

def normalize_bank(series):
    return series.astype(str).str.strip().str.lstrip("0").replace("", "0")

# Build keys from the node table's bank and account components.
node_bank = normalize_bank(nodes["bank"])
node_account = nodes["acct_key"].str.split("_", n=1).str[1]
node_keys = set(node_bank + "_" + node_account)

chunk = pd.read_csv(
    DATA,
    nrows=1000,
    usecols=["Timestamp", "From Bank", "Account", "To Bank", "Account.1"],
    dtype={
        "From Bank": str,
        "Account": str,
        "To Bank": str,
        "Account.1": str
    }
)

chunk["Timestamp"] = pd.to_datetime(
    chunk["Timestamp"], format="%Y/%m/%d %H:%M"
)
chunk = chunk[chunk["Timestamp"] < "2022-09-04"]

sender_keys = (
    normalize_bank(chunk["From Bank"]) + "_" + chunk["Account"].str.strip()
)
receiver_keys = (
    normalize_bank(chunk["To Bank"]) + "_" + chunk["Account.1"].str.strip()
)

print("Normalized node key examples:", list(node_keys)[:5])
print("Normalized sender examples:", sender_keys.head(5).tolist())
print("Matching sender keys:", sender_keys.isin(node_keys).sum(), "/", len(sender_keys))
print("Matching receiver keys:", receiver_keys.isin(node_keys).sum(), "/", len(receiver_keys))

Normalized node key examples: ['13327_8019BFA30', '243614_810C71160', '112989_8050ABDE0', '32269_8030CAF60', '32508_80B7999E0']
Normalized sender examples: ['10_8000EBD30', '3208_8000F4580', '3209_8000F4670', '12_8000F5030', '10_8000F5200']
Matching sender keys: 1000 / 1000
Matching receiver keys: 1000 / 1000


In [39]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
SAMPLE_DIR = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(SAMPLE_DIR / "nodes.csv", dtype={"acct_key": str})
n_nodes = len(nodes)

def normalize_bank(series):
    return series.astype(str).str.strip().str.lstrip("0").replace("", "0")

node_account = nodes["acct_key"].str.split("_", n=1).str[1]
node_keys = normalize_bank(nodes["bank"]) + "_" + node_account
node_lookup = dict(zip(node_keys, nodes["node_id"].astype(np.int64)))

# Per-account accumulators
in_count = np.zeros(n_nodes, dtype=np.int64)
out_count = np.zeros(n_nodes, dtype=np.int64)
in_total = np.zeros(n_nodes, dtype=np.float64)
out_total = np.zeros(n_nodes, dtype=np.float64)
in_max = np.zeros(n_nodes, dtype=np.float64)
out_max = np.zeros(n_nodes, dtype=np.float64)

unique_pairs_in = []
unique_pairs_out = []
account_days = []
account_formats = []

usecols = [
    "Timestamp", "From Bank", "Account", "To Bank", "Account.1",
    "Amount Received", "Amount Paid", "Payment Format"
]

processed = 0

for chunk in pd.read_csv(DATA, chunksize=100_000, usecols=usecols):
    ts = pd.to_datetime(
        chunk["Timestamp"], format="%Y/%m/%d %H:%M", errors="raise"
    )
    keep = ts < pd.Timestamp("2022-09-04")
    if not keep.any():
        continue

    chunk = chunk.loc[keep].copy()
    ts = ts.loc[keep]

    sender_key = (
        normalize_bank(chunk["From Bank"]) + "_" +
        chunk["Account"].astype(str).str.strip()
    )
    receiver_key = (
        normalize_bank(chunk["To Bank"]) + "_" +
        chunk["Account.1"].astype(str).str.strip()
    )

    src = sender_key.map(node_lookup)
    dst = receiver_key.map(node_lookup)

    # Ignore nothing silently: every selected transaction must map to nodes.
    if src.isna().any() or dst.isna().any():
        raise ValueError("Unmapped account key found in the 3-day sample.")

    src = src.to_numpy(dtype=np.int64)
    dst = dst.to_numpy(dtype=np.int64)
    paid = chunk["Amount Paid"].to_numpy(dtype=np.float64)
    received = chunk["Amount Received"].to_numpy(dtype=np.float64)
    fmt = chunk["Payment Format"].astype("category").cat.codes.to_numpy()
    day = (ts.astype("datetime64[ns]").astype("int64") // 1_000_000_000) // 86400

    np.add.at(out_count, src, 1)
    np.add.at(in_count, dst, 1)
    np.add.at(out_total, src, paid)
    np.add.at(in_total, dst, received)
    np.maximum.at(out_max, src, paid)
    np.maximum.at(in_max, dst, received)

    unique_pairs_out.append(np.column_stack((src, dst)))
    unique_pairs_in.append(np.column_stack((dst, src)))
    account_days.append(np.vstack((
        np.column_stack((src, day.to_numpy())),
        np.column_stack((dst, day.to_numpy()))
    )))
    account_formats.append(np.vstack((
        np.column_stack((src, fmt)),
        np.column_stack((dst, fmt))
    )))

    processed += len(chunk)

# Deduplicate transaction counterparties and account-day / account-format pairs.
pairs_out = np.unique(np.vstack(unique_pairs_out), axis=0)
pairs_in = np.unique(np.vstack(unique_pairs_in), axis=0)
unique_out = np.bincount(pairs_out[:, 0], minlength=n_nodes)
unique_in = np.bincount(pairs_in[:, 0], minlength=n_nodes)

days_unique = np.unique(np.vstack(account_days), axis=0)
active_days = np.bincount(days_unique[:, 0].astype(np.int64), minlength=n_nodes)

formats_unique = np.unique(np.vstack(account_formats), axis=0)
format_count = np.bincount(formats_unique[:, 0].astype(np.int64), minlength=n_nodes)

in_mean = np.divide(in_total, in_count, out=np.zeros(n_nodes), where=in_count > 0)
out_mean = np.divide(out_total, out_count, out=np.zeros(n_nodes), where=out_count > 0)

features = pd.DataFrame({
    "in_count": in_count,
    "out_count": out_count,
    "unique_in_counterparties": unique_in,
    "unique_out_counterparties": unique_out,
    "in_amount_total": in_total,
    "out_amount_total": out_total,
    "in_amount_mean": in_mean,
    "out_amount_mean": out_mean,
    "in_amount_max": in_max,
    "out_amount_max": out_max,
    "active_days": active_days,
    "distinct_payment_formats": format_count,
})

assert len(features) == len(nodes)
assert processed == 2_076_752, f"Unexpected sample transaction count: {processed}"

out_file = SAMPLE_DIR / "account_features_base.csv"
features.insert(0, "node_id", nodes["node_id"])
features.to_csv(out_file, index=False)

print("Transactions processed:", processed)
print("Feature matrix shape (including node_id):", features.shape)
print("Missing values:", int(features.isna().sum().sum()))
print("Saved to:", out_file)
print(features.head())

Transactions processed: 2076752
Feature matrix shape (including node_id): (510772, 13)
Missing values: 0
Saved to: /home/ibm-user/shared/data/processed_sample/account_features_base.csv
   node_id  in_count  out_count  unique_in_counterparties  \
0        0         9         53                         1   
1        1        11         78                         2   
2        2         6         45                         1   
3        3         5         35                         4   
4        4         9         44                         1   

   unique_out_counterparties  in_amount_total  out_amount_total  \
0                         13         39747.80      7.095937e+05   
1                         13        963224.73      1.439863e+07   
2                          9       3104529.72      1.588054e+08   
3                         10          5845.82      1.252588e+05   
4                          9         34487.19      2.777879e+07   

   in_amount_mean  out_amount_mean  in_amount

In [1]:
from pathlib import Path

base = Path("/home/ibm-user/shared/data/processed_sample")

for name in ["nodes.csv", "edges.npz", "account_features_base.csv"]:
    p = base / name
    print(name, "exists:", p.exists(),
          "size (MB):", round(p.stat().st_size / 1_000_000, 2) if p.exists() else "-")

nodes.csv exists: True size (MB): 19.98
edges.npz exists: True size (MB): 19.04
account_features_base.csv exists: True size (MB): 39.58


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(BASE / "nodes.csv")
features = pd.read_csv(BASE / "account_features_base.csv")

print("Node rows:", len(nodes))
print("Feature rows:", len(features))
print("Feature columns:", features.columns.tolist())
print("Duplicate node IDs:", int(features["node_id"].duplicated().sum()))
print("Missing values:", int(features.isna().sum().sum()))
print("Feature row IDs match node IDs:",
      np.array_equal(features["node_id"].to_numpy(), nodes["node_id"].to_numpy()))
print("\nFirst 5 rows:")
print(features.head().to_string(index=False))

Node rows: 510772
Feature rows: 510772
Feature columns: ['node_id', 'in_count', 'out_count', 'unique_in_counterparties', 'unique_out_counterparties', 'in_amount_total', 'out_amount_total', 'in_amount_mean', 'out_amount_mean', 'in_amount_max', 'out_amount_max', 'active_days', 'distinct_payment_formats']
Duplicate node IDs: 0
Missing values: 0
Feature row IDs match node IDs: True

First 5 rows:
 node_id  in_count  out_count  unique_in_counterparties  unique_out_counterparties  in_amount_total  out_amount_total  in_amount_mean  out_amount_mean  in_amount_max  out_amount_max  active_days  distinct_payment_formats
       0         9         53                         1                         13         39747.80         709593.68     4416.422222     1.338856e+04       38738.05       177848.61            3                         5
       1        11         78                         2                         13        963224.73       14398633.63    87565.884545     1.845979e+05      947389

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/home/ibm-user/shared/data/processed_sample")
features = pd.read_csv(BASE / "account_features_base.csv")

with np.load(BASE / "edges.npz") as z:
    src = z["src"].astype(np.int64)
    dst = z["dst"].astype(np.int64)
    amount = z["amount"].astype(np.float64)
    fmt = z["fmt"].astype(np.int64)

n_nodes = len(features)

# Provisional ratio: based on the saved base totals.
# We'll replace it if raw Amount Paid/Received totals differ.
features["pass_through_ratio"] = np.divide(
    features["out_amount_total"].to_numpy(),
    features["in_amount_total"].to_numpy(),
    out=np.zeros(n_nodes, dtype=np.float64),
    where=features["in_amount_total"].to_numpy() > 0
)

# ACH and Cash codes in the saved archive: ACH=0, Cash=2.
is_ach_cash = np.isin(fmt, [0, 2]).astype(np.int64)

ach_cash_incidence = np.bincount(src, weights=is_ach_cash, minlength=n_nodes)
ach_cash_incidence += np.bincount(dst, weights=is_ach_cash, minlength=n_nodes)

transaction_incidence = (
    features["in_count"].to_numpy() + features["out_count"].to_numpy()
)

features["ach_cash_share"] = np.divide(
    ach_cash_incidence,
    transaction_incidence,
    out=np.zeros(n_nodes, dtype=np.float64),
    where=transaction_incidence > 0
)

out_path = BASE / "account_features_sample_v1.csv"
features.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", features.shape)
print("Missing values:", int(features.isna().sum().sum()))
print("Pass-through ratio finite:",
      bool(np.isfinite(features["pass_through_ratio"]).all()))
print("ACH/cash share range:",
      float(features["ach_cash_share"].min()),
      "to", float(features["ach_cash_share"].max()))
print(features[[
    "node_id", "pass_through_ratio", "ach_cash_share"
]].head().to_string(index=False))

Saved: /home/ibm-user/shared/data/processed_sample/account_features_sample_v1.csv
Shape: (510772, 15)
Missing values: 0
Pass-through ratio finite: True
ACH/cash share range: 0.0 to 1.0
 node_id  pass_through_ratio  ach_cash_share
       0           17.852401        0.209677
       1           14.948364        0.393258
       2           51.152790        0.254902
       3           21.427069        0.175000
       4          805.481336        0.150943


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
BASE = Path("/home/ibm-user/shared/data/processed_sample")

raw = pd.read_csv(
    DATA,
    nrows=10_000,
    usecols=["Timestamp", "Amount Received", "Amount Paid"]
)
raw["Timestamp"] = pd.to_datetime(
    raw["Timestamp"], format="%Y/%m/%d %H:%M"
)
raw = raw[raw["Timestamp"] < "2022-09-04"]

with np.load(BASE / "edges.npz") as z:
    saved_amount = z["amount"][:len(raw)].astype(np.float64)

paid = raw["Amount Paid"].to_numpy(dtype=np.float64)
received = raw["Amount Received"].to_numpy(dtype=np.float64)

print("Rows compared:", len(raw))
print("Saved amount equals Amount Paid:", np.allclose(saved_amount, paid))
print("Saved amount equals Amount Received:", np.allclose(saved_amount, received))
print("Max abs difference vs paid:", float(np.max(np.abs(saved_amount - paid))))
print("Max abs difference vs received:", float(np.max(np.abs(saved_amount - received))))

Rows compared: 10000
Saved amount equals Amount Paid: True
Saved amount equals Amount Received: False
Max abs difference vs paid: 27.0
Max abs difference vs received: 27.0


In [5]:
from pathlib import Path
import pandas as pd

base = Path("/home/ibm-user/shared/data/processed_sample")
p = base / "account_features_base.csv"

print("Feature file exists:", p.exists())
print("Feature file size (MB):", round(p.stat().st_size / 1_000_000, 2))

features = pd.read_csv(p, nrows=5)
print("Columns:", features.columns.tolist())
print(features[[
    "node_id", "in_amount_total", "out_amount_total",
    "in_amount_mean", "out_amount_mean"
]].to_string(index=False))


Feature file exists: True
Feature file size (MB): 39.58
Columns: ['node_id', 'in_count', 'out_count', 'unique_in_counterparties', 'unique_out_counterparties', 'in_amount_total', 'out_amount_total', 'in_amount_mean', 'out_amount_mean', 'in_amount_max', 'out_amount_max', 'active_days', 'distinct_payment_formats']
 node_id  in_amount_total  out_amount_total  in_amount_mean  out_amount_mean
       0         39747.80         709593.68     4416.422222     1.338856e+04
       1        963224.73       14398633.63    87565.884545     1.845979e+05
       2       3104529.72      158805355.28   517421.620000     3.529008e+06
       3          5845.82         125258.79     1169.164000     3.578823e+03
       4         34487.19       27778787.87     3831.910000     6.313361e+05


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
BASE = Path("/home/ibm-user/shared/data/processed_sample")

nodes = pd.read_csv(BASE / "nodes.csv", dtype={"acct_key": str})
n_nodes = len(nodes)

def normalize_bank(series):
    return series.astype(str).str.strip().str.lstrip("0").replace("", "0")

node_account = nodes["acct_key"].str.split("_", n=1).str[1]
node_keys = normalize_bank(nodes["bank"]) + "_" + node_account
node_lookup = dict(zip(node_keys, nodes["node_id"].astype(np.int64)))

parts = []
processed = 0
usecols = [
    "Timestamp", "From Bank", "Account", "To Bank", "Account.1",
    "Amount Received", "Amount Paid"
]

for chunk in pd.read_csv(DATA, chunksize=100_000, usecols=usecols):
    ts = pd.to_datetime(chunk["Timestamp"], format="%Y/%m/%d %H:%M")
    keep = ts < pd.Timestamp("2022-09-04")
    if not keep.any():
        continue

    chunk = chunk.loc[keep].copy()
    src_key = normalize_bank(chunk["From Bank"]) + "_" + chunk["Account"].astype(str).str.strip()
    dst_key = normalize_bank(chunk["To Bank"]) + "_" + chunk["Account.1"].astype(str).str.strip()

    chunk["src"] = src_key.map(node_lookup)
    chunk["dst"] = dst_key.map(node_lookup)

    if chunk[["src", "dst"]].isna().any().any():
        raise ValueError("Unmapped account key found.")

    chunk["src"] = chunk["src"].astype(np.int64)
    chunk["dst"] = chunk["dst"].astype(np.int64)

    outgoing = chunk.groupby("src").agg(
        out_count=("Amount Paid", "size"),
        out_total=("Amount Paid", "sum"),
        out_mean=("Amount Paid", "mean"),
        out_max=("Amount Paid", "max")
    ).rename_axis("node_id")

    incoming = chunk.groupby("dst").agg(
        in_count=("Amount Received", "size"),
        in_total=("Amount Received", "sum"),
        in_mean=("Amount Received", "mean"),
        in_max=("Amount Received", "max")
    ).rename_axis("node_id")

    parts.append((outgoing, incoming))
    processed += len(chunk)

assert processed == 2_076_752, f"Unexpected sample rows: {processed}"

# Merge the per-chunk summaries, weighting means by transaction counts.
out_frames = [p[0] for p in parts]
in_frames = [p[1] for p in parts]

out_sum = pd.concat(out_frames).groupby(level=0).agg({
    "out_count": "sum",
    "out_total": "sum",
    "out_max": "max"
})
out_weighted = pd.concat([
    df[["out_count"]].join(df[["out_mean"]].mul(df["out_count"], axis=0).rename(columns={"out_mean":"weighted"}))
    for df in out_frames
]).groupby(level=0).sum()
out_sum["out_mean"] = out_weighted["weighted"] / out_sum["out_count"]

in_sum = pd.concat(in_frames).groupby(level=0).agg({
    "in_count": "sum",
    "in_total": "sum",
    "in_max": "max"
})
in_weighted = pd.concat([
    df[["in_count"]].join(df[["in_mean"]].mul(df["in_count"], axis=0).rename(columns={"in_mean":"weighted"}))
    for df in in_frames
]).groupby(level=0).sum()
in_sum["in_mean"] = in_weighted["weighted"] / in_sum["in_count"]

result = pd.DataFrame({"node_id": np.arange(n_nodes)})
result = result.merge(in_sum, left_on="node_id", right_index=True, how="left")
result = result.merge(out_sum, left_on="node_id", right_index=True, how="left")

for col in ["in_count", "out_count"]:
    result[col] = result[col].fillna(0).astype(np.int64)
for col in ["in_total", "in_mean", "in_max", "out_total", "out_mean", "out_max"]:
    result[col] = result[col].fillna(0.0)

out_file = BASE / "account_amount_features_corrected.csv"
result.to_csv(out_file, index=False)

print("Rows processed:", processed)
print("Result shape:", result.shape)
print("Saved:", out_file)
print(result.head().to_string(index=False))

Rows processed: 2076752
Result shape: (510772, 9)
Saved: /home/ibm-user/shared/data/processed_sample/account_amount_features_corrected.csv
 node_id  in_count   in_total     in_max       in_mean  out_count    out_total      out_max     out_mean
       0         9   39747.80   38738.05   4416.422222         53    709593.68    177848.61 1.338856e+04
       1        11  963224.73  947389.86  87565.884545         78  14398633.63  11086012.22 1.845979e+05
       2         6 3104529.72 3079734.02 517421.620000         45 158805355.28 105891407.47 3.529008e+06
       3         5    5845.82    5439.01   1169.164000         35    125258.79     22443.26 3.578823e+03
       4         9   34487.19   20910.84   3831.910000         44  27778787.87  11701716.74 6.313361e+05


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("/home/ibm-user/shared/data/processed_sample")

features = pd.read_csv(base / "account_features_base.csv")
amounts = pd.read_csv(base / "account_amount_features_corrected.csv")

assert len(features) == len(amounts) == 510772
assert features["node_id"].equals(amounts["node_id"]), "Node order mismatch!"

# Replace amount-related fields with the corrected directional aggregates.
for col in [
    "in_count", "out_count",
    "in_amount_total", "out_amount_total",
    "in_amount_mean", "out_amount_mean",
    "in_amount_max", "out_amount_max"
]:
    source_col = {
        "in_amount_total": "in_total",
        "out_amount_total": "out_total",
        "in_amount_mean": "in_mean",
        "out_amount_mean": "out_mean",
        "in_amount_max": "in_max",
        "out_amount_max": "out_max",
    }.get(col, col)

    features[col] = amounts[source_col].to_numpy()

out = base / "account_features_base_corrected.csv"
features.to_csv(out, index=False)

print("Saved:", out)
print("Shape:", features.shape)
print("Duplicate node IDs:", features["node_id"].duplicated().sum())
print("Missing values:", int(features.isna().sum().sum()))
print(features[[
    "node_id", "in_count", "out_count",
    "in_amount_total", "out_amount_total",
    "in_amount_mean", "out_amount_mean"
]].head().to_string(index=False))

Saved: /home/ibm-user/shared/data/processed_sample/account_features_base_corrected.csv
Shape: (510772, 13)
Duplicate node IDs: 0
Missing values: 0
 node_id  in_count  out_count  in_amount_total  out_amount_total  in_amount_mean  out_amount_mean
       0         9         53         39747.80         709593.68     4416.422222     1.338856e+04
       1        11         78        963224.73       14398633.63    87565.884545     1.845979e+05
       2         6         45       3104529.72      158805355.28   517421.620000     3.529008e+06
       3         5         35          5845.82         125258.79     1169.164000     3.578823e+03
       4         9         44         34487.19       27778787.87     3831.910000     6.313361e+05


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("/home/ibm-user/shared/data/processed_sample")

features = pd.read_csv(base / "account_features_base_corrected.csv")
nodes = pd.read_csv(base / "nodes.csv")
edges = np.load(base / "edges.npz")

# Confirm node order and edge alignment.
assert len(features) == len(nodes) == 510772
assert np.array_equal(
    features["node_id"].to_numpy(),
    nodes["node_id"].to_numpy()
)

src = edges["src"]
dst = edges["dst"]
fmt = edges["fmt"]

n = len(nodes)
assert len(src) == len(dst) == len(fmt) == len(edges["amount"])

# Transaction-incidence counts: each transaction counts once for each endpoint.
incidence_count = np.bincount(
    np.concatenate([src, dst]),
    minlength=n
)

# Payment-format codes established during sample processing:
# ACH = 0, Cash = 2.
ach_cash_incidence = np.bincount(
    np.concatenate([src[ np.isin(fmt, [0, 2]) ],
                    dst[ np.isin(fmt, [0, 2]) ]]),
    minlength=n
)

features["pass_through_ratio"] = np.divide(
    features["out_amount_total"].to_numpy(),
    features["in_amount_total"].to_numpy(),
    out=np.zeros(n, dtype=float),
    where=features["in_amount_total"].to_numpy() > 0
)

features["ach_cash_share"] = np.divide(
    ach_cash_incidence,
    incidence_count,
    out=np.zeros(n, dtype=float),
    where=incidence_count > 0
)

out = base / "account_features_sample_v2.csv"
features.to_csv(out, index=False)

print("Saved:", out)
print("Shape:", features.shape)
print("Missing values:", int(features.isna().sum().sum()))
print("Pass-through ratio finite:", np.isfinite(features["pass_through_ratio"]).all())
print("ACH/Cash share range:",
      float(features["ach_cash_share"].min()),
      float(features["ach_cash_share"].max()))
print("Accounts with zero incoming amount:",
      int((features["in_amount_total"] == 0).sum()))
print(features[[
    "node_id", "in_amount_total", "out_amount_total",
    "pass_through_ratio", "ach_cash_share"
]].head().to_string(index=False))

Saved: /home/ibm-user/shared/data/processed_sample/account_features_sample_v2.csv
Shape: (510772, 15)
Missing values: 0
Pass-through ratio finite: True
ACH/Cash share range: 0.0 1.0
Accounts with zero incoming amount: 91964
 node_id  in_amount_total  out_amount_total  pass_through_ratio  ach_cash_share
       0         39747.80         709593.68           17.852401        0.209677
       1        963224.73       14398633.63           14.948364        0.393258
       2       3104529.72      158805355.28           51.152790        0.254902
       3          5845.82         125258.79           21.427069        0.175000
       4         34487.19       27778787.87          805.481336        0.150943


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("/home/ibm-user/shared/data/processed_sample")

features = pd.read_csv(base / "account_features_sample_v2.csv")
nodes = pd.read_csv(base / "nodes.csv")

print("Feature columns:")
print(features.columns.tolist())

print("\nUnexpected label/split columns in features:")
print([
    c for c in features.columns
    if any(term in c.lower() for term in ["label", "launder", "split", "is_mule"])
])

print("\nPass-through ratio percentiles:")
print(
    features["pass_through_ratio"]
    .quantile([0, 0.5, 0.9, 0.95, 0.99, 0.999, 1])
    .to_string()
)

print("\nLargest pass-through ratios:")
print(
    features.nlargest(10, "pass_through_ratio")[
        ["node_id", "in_amount_total", "out_amount_total", "pass_through_ratio"]
    ].to_string(index=False)
)

print("\nFeature/node ID alignment:",
      np.array_equal(features["node_id"].to_numpy(), nodes["node_id"].to_numpy()))

print("\nLabel distribution (kept in nodes, not features):")
print(nodes["label"].value_counts(dropna=False).sort_index().to_string())

print("\nSplit distribution:")
print(nodes.groupby("split")["label"].agg(["count", "sum"]).to_string())

Feature columns:
['node_id', 'in_count', 'out_count', 'unique_in_counterparties', 'unique_out_counterparties', 'in_amount_total', 'out_amount_total', 'in_amount_mean', 'out_amount_mean', 'in_amount_max', 'out_amount_max', 'active_days', 'distinct_payment_formats', 'pass_through_ratio', 'ach_cash_share']

Unexpected label/split columns in features:
[]

Pass-through ratio percentiles:
0.000    0.000000e+00
0.500    9.150158e-01
0.900    3.730209e+00
0.950    2.143057e+01
0.990    8.145816e+02
0.999    1.625140e+05
1.000    2.538633e+10

Largest pass-through ratios:
 node_id  in_amount_total  out_amount_total  pass_through_ratio
   22758             0.07      1.777043e+09        2.538633e+10
  386135             0.05      1.204999e+08        2.409998e+09
  121062             0.09      1.743652e+08        1.937392e+09
   26480             0.02      3.681180e+07        1.840590e+09
  350942             0.01      1.422313e+07        1.422313e+09
   30114             0.28      3.553362e+08   

In [10]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

raw = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")

cols = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1"
]

t0 = time.time()
df = pd.read_csv(raw, usecols=cols, nrows=200_000, dtype=str)

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce"
)

# Normalize bank IDs to match nodes.csv account keys.
from_bank = df["From Bank"].str.strip().str.lstrip("0").replace("", "0")
to_bank = df["To Bank"].str.strip().str.lstrip("0").replace("", "0")

df["src_key"] = from_bank + "_" + df["Account"].str.strip()
df["dst_key"] = to_bank + "_" + df["Account.1"].str.strip()

# Build incoming receive events and outgoing send events.
incoming = df[["dst_key", "Timestamp"]].rename(
    columns={"dst_key": "acct_key", "Timestamp": "receive_time"}
)
outgoing = df[["src_key", "Timestamp"]].rename(
    columns={"src_key": "acct_key", "Timestamp": "send_time"}
)

incoming = incoming.dropna().sort_values(["acct_key", "receive_time"])
outgoing = outgoing.dropna().sort_values(["acct_key", "send_time"])

# For each receive event, find the next send by that account.
matched = pd.merge_asof(
    incoming.sort_values("receive_time"),
    outgoing.sort_values("send_time"),
    left_on="receive_time",
    right_on="send_time",
    by="acct_key",
    direction="forward",
    allow_exact_matches=False
)

elapsed = time.time() - t0
valid = matched["send_time"].notna().sum()

print("Rows benchmarked:", len(df))
print("Receive events:", len(incoming))
print("Receive events with a later send:", int(valid))
print("Elapsed seconds:", round(elapsed, 2))
print("Elapsed minutes:", round(elapsed / 60, 2))

Rows benchmarked: 200000
Receive events: 200000
Receive events with a later send: 38725
Elapsed seconds: 0.94
Elapsed minutes: 0.02


In [11]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

raw = Path("/home/ibm-user/shared/data/HI-Small_Trans.csv")
base = Path("/home/ibm-user/shared/data/processed_sample")
nodes = pd.read_csv(base / "nodes.csv", dtype={"acct_key": str})

node_keys = nodes["acct_key"].astype(str).to_numpy()
node_lookup = dict(zip(node_keys, nodes["node_id"].to_numpy(dtype=np.int64)))
n_nodes = len(nodes)

cols = [
    "Timestamp", "From Bank", "Account",
    "To Bank", "Account.1"
]

t0 = time.time()
incoming_parts = []
outgoing_parts = []
rows_kept = 0

for chunk in pd.read_csv(raw, usecols=cols, dtype=str, chunksize=100_000):
    ts = pd.to_datetime(
        chunk["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce"
    )
    keep = ts < pd.Timestamp("2022-09-04")
    if not keep.any():
        continue

    chunk = chunk.loc[keep].copy()
    ts = ts.loc[keep]

    from_bank = chunk["From Bank"].str.strip().str.lstrip("0").replace("", "0")
    to_bank = chunk["To Bank"].str.strip().str.lstrip("0").replace("", "0")

    src_key = from_bank + "_" + chunk["Account"].str.strip()
    dst_key = to_bank + "_" + chunk["Account.1"].str.strip()

    src = src_key.map(node_lookup)
    dst = dst_key.map(node_lookup)

    if src.isna().any() or dst.isna().any():
        raise ValueError("Account key did not map to sample nodes.")

    incoming_parts.append(pd.DataFrame({
        "node_id": dst.to_numpy(dtype=np.int64),
        "time": ts.astype("int64").to_numpy()
    }))
    outgoing_parts.append(pd.DataFrame({
        "node_id": src.to_numpy(dtype=np.int64),
        "time": ts.astype("int64").to_numpy()
    }))
    rows_kept += len(chunk)

    if time.time() - t0 > 600:
        raise TimeoutError("Exceeded the 10-minute feature budget.")

print("Rows retained:", rows_kept)

incoming = pd.concat(incoming_parts, ignore_index=True)
outgoing = pd.concat(outgoing_parts, ignore_index=True)
del incoming_parts, outgoing_parts

incoming = incoming.sort_values(["time", "node_id"]).reset_index(drop=True)
outgoing = outgoing.sort_values(["time", "node_id"]).reset_index(drop=True)

# Match each incoming event to the next strictly later outgoing event
# for the same account.
matched = pd.merge_asof(
    incoming.sort_values("time"),
    outgoing.sort_values("time"),
    on="time",
    by="node_id",
    direction="forward",
    allow_exact_matches=False,
    suffixes=("_receive", "_send")
)

matched["minutes_to_next_send"] = (
    matched["time_send"] - matched["time_receive"]
) / 60_000_000_000

valid = matched.dropna(subset=["minutes_to_next_send"])
median_timing = valid.groupby("node_id")["minutes_to_next_send"].median()

timing = pd.DataFrame({"node_id": np.arange(n_nodes)})
timing["median_receive_to_next_send_min"] = (
    timing["node_id"].map(median_timing).fillna(0.0)
)

out = base / "account_timing_sample.csv"
timing.to_csv(out, index=False)

elapsed = time.time() - t0
print("Elapsed minutes:", round(elapsed / 60, 2))
print("Accounts with matched timing:", int(median_timing.index.nunique()))
print("Saved:", out)
print(timing["median_receive_to_next_send_min"].describe().to_string())

Rows retained: 2076752


KeyError: 'time_send'

In [12]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

base = Path("/home/ibm-user/shared/data/processed_sample")
t0 = time.time()

# Give receive and send timestamps distinct names.
incoming_fix = incoming.rename(columns={"time": "receive_time"})
outgoing_fix = outgoing.rename(columns={"time": "send_time"})

matched = pd.merge_asof(
    incoming_fix.sort_values("receive_time"),
    outgoing_fix.sort_values("send_time"),
    left_on="receive_time",
    right_on="send_time",
    by="node_id",
    direction="forward",
    allow_exact_matches=False
)

matched["minutes_to_next_send"] = (
    matched["send_time"] - matched["receive_time"]
) / 60_000_000_000

valid = matched.dropna(subset=["minutes_to_next_send"])
median_timing = valid.groupby("node_id")["minutes_to_next_send"].median()

timing = pd.DataFrame({"node_id": np.arange(len(nodes))})
timing["median_receive_to_next_send_min"] = (
    timing["node_id"].map(median_timing).fillna(0.0)
)

out = base / "account_timing_sample.csv"
timing.to_csv(out, index=False)

print("Elapsed minutes for matching and aggregation:", round((time.time() - t0) / 60, 2))
print("Accounts with matched timing:", int(median_timing.index.nunique()))
print("Saved:", out)
print(timing["median_receive_to_next_send_min"].describe().to_string())

Elapsed minutes for matching and aggregation: 0.02
Accounts with matched timing: 256579
Saved: /home/ibm-user/shared/data/processed_sample/account_timing_sample.csv
count    510772.000000
mean          0.282490
std           0.481711
min           0.000000
25%           0.000000
50%           0.001000
75%           0.427500
max           4.302000


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

base = Path("/home/ibm-user/shared/data/processed_sample")

features = pd.read_csv(base / "account_features_sample_v2.csv")
timing = pd.read_csv(base / "account_timing_sample.csv")
nodes = pd.read_csv(base / "nodes.csv")

assert features["node_id"].equals(timing["node_id"]), "Feature/timing node order mismatch"
assert features["node_id"].equals(nodes["node_id"]), "Feature/node order mismatch"

features["median_receive_to_next_send_min"] = (
    timing["median_receive_to_next_send_min"].to_numpy()
)

# Explicitly exclude identifiers and targets from the model input columns.
excluded = {"node_id", "label", "split", "is_laundering", "is_mule"}
feature_cols = [c for c in features.columns if c not in excluded]
X = features[feature_cols].to_numpy(dtype=np.float32)

assert X.shape == (len(nodes), len(feature_cols))
assert np.isfinite(X).all(), "Non-finite values found"
assert "label" not in feature_cols
assert "split" not in feature_cols

out_csv = base / "account_features_sample_v3.csv"
out_npy = base / "X_sample_raw.npy"
out_names = base / "feature_names_sample.txt"

features.to_csv(out_csv, index=False)
np.save(out_npy, X)
out_names.write_text("\n".join(feature_cols) + "\n")

print("Feature table:", out_csv)
print("Raw matrix:", out_npy, X.shape, X.dtype)
print("Feature names:", out_names)
print("Columns:", feature_cols)
print("All finite:", np.isfinite(X).all())
print("Timing zeros:", int((features["median_receive_to_next_send_min"] == 0).sum()))

Feature table: /home/ibm-user/shared/data/processed_sample/account_features_sample_v3.csv
Raw matrix: /home/ibm-user/shared/data/processed_sample/X_sample_raw.npy (510772, 15) float32
Feature names: /home/ibm-user/shared/data/processed_sample/feature_names_sample.txt
Columns: ['in_count', 'out_count', 'unique_in_counterparties', 'unique_out_counterparties', 'in_amount_total', 'out_amount_total', 'in_amount_mean', 'out_amount_mean', 'in_amount_max', 'out_amount_max', 'active_days', 'distinct_payment_formats', 'pass_through_ratio', 'ach_cash_share', 'median_receive_to_next_send_min']
All finite: True
Timing zeros: 254193


In [14]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

base = Path("/home/ibm-user/shared/data/processed_sample")

X = np.load(base / "X_sample_raw.npy")
feature_names = (base / "feature_names_sample.txt").read_text().splitlines()
nodes = pd.read_csv(base / "nodes.csv")

assert X.shape == (len(nodes), len(feature_names))
assert list(nodes["split"].unique()) == ["train", "test", "val"] or set(nodes["split"]) == {"train", "val", "test"}

# Transform nonnegative count and amount features with log1p.
log_features = [
    "in_count", "out_count",
    "unique_in_counterparties", "unique_out_counterparties",
    "in_amount_total", "out_amount_total",
    "in_amount_mean", "out_amount_mean",
    "in_amount_max", "out_amount_max",
    "active_days", "distinct_payment_formats"
]

X_transformed = X.copy()
for name in log_features:
    j = feature_names.index(name)
    if np.any(X_transformed[:, j] < 0):
        raise ValueError(f"Negative values found in {name}; cannot apply log1p.")
    X_transformed[:, j] = np.log1p(X_transformed[:, j])

# Standardize using training rows only.
train_mask = nodes["split"].eq("train").to_numpy()
mu = X_transformed[train_mask].mean(axis=0, dtype=np.float64)
sigma = X_transformed[train_mask].std(axis=0, dtype=np.float64)
sigma[sigma == 0] = 1.0

X_standardized = ((X_transformed - mu) / sigma).astype(np.float32)

np.save(base / "X_sample_standardized.npy", X_standardized)
np.savez(base / "X_sample_scaler.npz", mean=mu, scale=sigma)
(base / "feature_names_sample.json").write_text(
    json.dumps(feature_names, indent=2) + "\n"
)

print("Saved standardized matrix:", X_standardized.shape, X_standardized.dtype)
print("Training rows:", int(train_mask.sum()))
print("All finite:", bool(np.isfinite(X_standardized).all()))
print("Training means (first 5):", X_standardized[train_mask].mean(axis=0)[:5])
print("Training stds (first 5):", X_standardized[train_mask].std(axis=0)[:5])
print("Saved scaler and feature names.")

Saved standardized matrix: (510772, 15) float32
Training rows: 306463
All finite: True
Training means (first 5): [2.5397888e-04 7.8257021e-05 9.9788456e-05 1.5370391e-04 1.0401117e-04]
Training stds (first 5): [1.0007532  0.9993898  0.99976104 1.0013449  1.0004596 ]
Saved scaler and feature names.


In [15]:
from pathlib import Path

shared = Path("/home/ibm-user/shared")
mulenet = shared / "mulenet"

print("mulenet directory exists:", mulenet.exists())

for name in ["build.py", "common.py"]:
    p = mulenet / name
    print(f"{name}: exists={p.exists()}", end="")
    if p.exists():
        print(f", size={p.stat().st_size:,} bytes")
    else:
        print()

mulenet directory exists: True
build.py: exists=True, size=5,840 bytes
common.py: exists=True, size=8,443 bytes


In [16]:
from pathlib import Path
import ast

shared = Path("/home/ibm-user/shared")
data = shared / "data"

for file_name, expected in [
    ("build.py", "build_processed"),
    ("common.py", "load_view"),
]:
    path = shared / "mulenet" / file_name
    source = path.read_text()
    tree = ast.parse(source)

    functions = {
        node.name for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    }

    print(f"\n{file_name}")
    print("Expected function present:", expected in functions)
    print("Functions:", sorted(functions))

print("\nRequired mapping file exists:",
      (data / "processed" / "bank_to_inst.csv").exists())

for folder in ["processed", "processed_sample"]:
    p = data / folder
    print(f"\n{folder} exists:", p.exists())
    if p.exists():
        print("Files:", sorted(x.name for x in p.iterdir()))
        


build.py
Expected function present: True
Functions: ['_chunks', 'build_processed']

common.py
Expected function present: True
Functions: ['_view_mask', 'account_features', 'data_dir', 'laundering_chains', 'load_edges', 'load_nodes', 'load_view']

Required mapping file exists: True

processed exists: True
Files: ['bank_to_inst.csv', 'edges.npz', 'edges_work.npy', 'node_labels.npy', 'nodes.csv']

processed_sample exists: True
Files: ['X_sample_raw.npy', 'X_sample_scaler.npz', 'X_sample_standardized.npy', 'account_amount_features_corrected.csv', 'account_features_base.csv', 'account_features_base_corrected.csv', 'account_features_sample_v1.csv', 'account_features_sample_v2.csv', 'account_features_sample_v3.csv', 'account_timing_sample.csv', 'edges.npz', 'feature_names_sample.json', 'feature_names_sample.txt', 'nodes.csv']


In [17]:
from pathlib import Path
import shutil

D = Path("/home/ibm-user/shared/data")
old = D / "old_vishwajeet"
old.mkdir(exist_ok=True)

# Move old full-data outputs, preserving bank_to_inst.csv.
for rel in [
    "processed/edges_work.npy",
    "processed/node_labels.npy",
    "processed/nodes.csv",
    "processed/edges.npz",
]:
    src = D / rel
    if src.exists():
        dest = old / rel.replace("/", "__")
        if dest.exists():
            print("Already exists, leaving source untouched:", dest.name)
        else:
            shutil.move(str(src), str(dest))
            print("Moved:", rel)

# Move all old sample outputs.
sample_dir = D / "processed_sample"
if sample_dir.exists():
    for src in list(sample_dir.iterdir()):
        dest = old / ("sample__" + src.name)
        if dest.exists():
            print("Already exists, leaving source untouched:", dest.name)
        else:
            shutil.move(str(src), str(dest))
            print("Moved sample file:", src.name)

print("\nBackup contents:")
print(*sorted(p.name for p in old.iterdir()), sep="\n")

print("\nFiles remaining in processed/:")
print(sorted(p.name for p in (D / "processed").iterdir()))

print("\nFiles remaining in processed_sample/:")
print(sorted(p.name for p in sample_dir.iterdir()) if sample_dir.exists() else "directory missing")


Moved: processed/edges_work.npy
Moved: processed/node_labels.npy
Moved: processed/nodes.csv
Moved: processed/edges.npz
Moved sample file: feature_names_sample.json
Moved sample file: account_features_sample_v1.csv
Moved sample file: account_features_base_corrected.csv
Moved sample file: account_amount_features_corrected.csv
Moved sample file: account_features_sample_v3.csv
Moved sample file: feature_names_sample.txt
Moved sample file: X_sample_standardized.npy
Moved sample file: nodes.csv
Moved sample file: account_features_base.csv
Moved sample file: X_sample_scaler.npz
Moved sample file: account_timing_sample.csv
Moved sample file: edges.npz
Moved sample file: X_sample_raw.npy
Moved sample file: account_features_sample_v2.csv

Backup contents:
processed__edges.npz
processed__edges_work.npy
processed__node_labels.npy
processed__nodes.csv
sample__X_sample_raw.npy
sample__X_sample_scaler.npz
sample__X_sample_standardized.npy
sample__account_amount_features_corrected.csv
sample__account_

In [18]:
import sys
from pathlib import Path

shared = Path("/home/ibm-user/shared")
sys.path.insert(0, str(shared))

from mulenet.build import build_processed

out = build_processed(sample_days=3)

print("\nReturned output directory:", out)
print("Output files:", sorted(p.name for p in Path(out).iterdir()))

accounts: 510,772
edges: 2,076,752 | laundering: 1,121 | mules: 1,680 (0.33%)
time range: 2022-09-01 00:00:00 -> 2022-09-03 23:59:00
accounts per member: {'B1': 91995, 'B2': 105675, 'B3': 103881, 'B4': 104001, 'B5': 105220}
label       0     1
split              
test   101819   336
train  305455  1008
val    101818   336
saved to /home/ibm-user/shared/data/processed_sample

Returned output directory: /home/ibm-user/shared/data/processed_sample
Output files: ['edges.npz', 'nodes.csv']


In [19]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, "/home/ibm-user/shared")

from mulenet.common import load_view, load_nodes

views = ["combined", "B1", "B2", "B3", "B4", "B5"]

for view in views:
    d = load_view(view, sample=True)
    print(
        view,
        "x:", tuple(d.x.shape),
        "directed edges:", d.edge_index_dir.shape[1],
        "features:", len(d.feature_names)
    )

nodes = load_nodes(sample=True)
x_combined = np.load(
    "/home/ibm-user/shared/data/processed_sample/x_combined.npy"
)
x_b1 = np.load(
    "/home/ibm-user/shared/data/processed_sample/x_B1.npy"
)

print("\nNode rows:", len(nodes))
print("Combined matrix shape:", x_combined.shape)
print("B1 matrix shape:", x_b1.shape)
print("Feature names:", d.feature_names)
print("Cached files:", sorted(
    p.name for p in Path("/home/ibm-user/shared/data/processed_sample").glob("x_*.npy")
))

/home/ibm-user/shared/mulenet/common.py:150: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  edge_index_dir=ei_dir, y=torch.from_numpy(nodes["label"].to_numpy(np.int64)),


combined x: (510772, 19) directed edges: 879982 features: 19
B1 x: (510772, 19) directed edges: 247699 features: 19
B2 x: (510772, 19) directed edges: 259636 features: 19
B3 x: (510772, 19) directed edges: 258803 features: 19
B4 x: (510772, 19) directed edges: 259041 features: 19
B5 x: (510772, 19) directed edges: 258688 features: 19

Node rows: 510772
Combined matrix shape: (510772, 19)
B1 matrix shape: (510772, 19)
Feature names: ['in_count', 'out_count', 'uniq_in', 'uniq_out', 'amt_in', 'amt_out', 'mean_in', 'mean_out', 'max_in', 'max_out', 'pass_through', 'first_in_to_last_out_h', 'receives_then_sends', 'active_days', 'n_formats', 'cross_member_share', 'share_ach', 'share_cash', 'in_view']
Cached files: ['x_B1.npy', 'x_B2.npy', 'x_B3.npy', 'x_B4.npy', 'x_B5.npy', 'x_combined.npy']


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

base = Path("/home/ibm-user/shared/data/processed_sample")
nodes = pd.read_csv(base / "nodes.csv")
with np.load(base / "edges.npz", allow_pickle=False) as z:
    edges = {k: z[k] for k in z.files}

print("Node columns:", nodes.columns.tolist())
print("Edge arrays:", sorted(edges.keys()))
print("Node count:", len(nodes))
print("Edge count:", len(edges["src"]))
print("Laundering transactions:", int(edges["is_laundering"].sum()))
print("Mule accounts:", int(nodes["label"].sum()))
print("Timestamp range:",
      pd.to_datetime(edges["ts"].min(), unit="s"),
      "->",
      pd.to_datetime(edges["ts"].max(), unit="s"))

print("\nSplit counts:")
print(pd.crosstab(nodes["split"], nodes["label"]).to_string())

print("\nCache checks:")
for view in ["combined", "B1", "B2", "B3", "B4", "B5"]:
    x = np.load(base / f"x_{view}.npy", mmap_mode="r")
    print(
        view,
        "shape=", x.shape,
        "dtype=", x.dtype,
        "all finite=", bool(np.isfinite(x).all())
    )

Node columns: ['node_idx', 'acct_key', 'bank', 'inst', 'label', 'split']
Edge arrays: ['amount', 'dst', 'fmt', 'fmt_names', 'is_laundering', 'src', 'ts']
Node count: 510772
Edge count: 2076752
Laundering transactions: 1121
Mule accounts: 1680
Timestamp range: 2022-09-01 00:00:00 -> 2022-09-03 23:59:00

Split counts:
label       0     1
split              
test   101819   336
train  305455  1008
val    101818   336

Cache checks:
combined shape= (510772, 19) dtype= float32 all finite= True
B1 shape= (510772, 19) dtype= float32 all finite= True
B2 shape= (510772, 19) dtype= float32 all finite= True
B3 shape= (510772, 19) dtype= float32 all finite= True
B4 shape= (510772, 19) dtype= float32 all finite= True
B5 shape= (510772, 19) dtype= float32 all finite= True


In [21]:
import sys
from pathlib import Path

sys.path.insert(0, "/home/ibm-user/shared")

from mulenet.build import build_processed

out = build_processed(sample_days=None)

print("\nReturned output directory:", out)
print("Output files:", sorted(p.name for p in Path(out).iterdir()))

accounts: 515,088
edges: 5,078,345 | laundering: 5,177 | mules: 6,357 (1.23%)
time range: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00
accounts per member: {'B1': 92779, 'B2': 106558, 'B3': 104799, 'B4': 104850, 'B5': 106102}
label       0     1
split              
test   101747  1272
train  305238  3814
val    101746  1271
saved to /home/ibm-user/shared/data/processed

Returned output directory: /home/ibm-user/shared/data/processed
Output files: ['bank_to_inst.csv', 'edges.npz', 'nodes.csv']


In [22]:
import numpy as np
from pathlib import Path
from mulenet.common import load_view, load_nodes, load_edges

base = Path("/home/ibm-user/shared/data/processed")

# Verify full-data nodes and edges
nodes = load_nodes(sample=False)
edges = load_edges(sample=False)

print("Nodes:", len(nodes))
print("Transactions:", len(edges["src"]))
print("Laundering transactions:", int(edges["is_laundering"].sum()))
print("Mule accounts:", int(nodes["label"].sum()))

# Generate features for the combined view and each institutional silo
for view in ["combined", "B1", "B2", "B3", "B4", "B5"]:
    data = load_view(view, sample=False)
    x = np.load(base / f"x_{view}.npy", mmap_mode="r")
    print(
        f"{view}: features={x.shape}, dtype={x.dtype}, "
        f"directed edges={data.edge_index_dir.shape[1]}, "
        f"finite={bool(np.isfinite(x).all())}"
    )

Nodes: 515088
Transactions: 5078345
Laundering transactions: 5177
Mule accounts: 6357
combined: features=(515088, 19), dtype=float32, directed edges=1015736, finite=True
B1: features=(515088, 19), dtype=float32, directed edges=290207, finite=True
B2: features=(515088, 19), dtype=float32, directed edges=309596, finite=True
B3: features=(515088, 19), dtype=float32, directed edges=308115, finite=True
B4: features=(515088, 19), dtype=float32, directed edges=308255, finite=True
B5: features=(515088, 19), dtype=float32, directed edges=308904, finite=True


In [23]:
import numpy as np
import pandas as pd
from pathlib import Path

base = Path("/home/ibm-user/shared/data/processed")

nodes = pd.read_csv(base / "nodes.csv")
with np.load(base / "edges.npz", allow_pickle=False) as z:
    edges = {k: z[k] for k in z.files}

required_node_cols = [
    "node_idx", "acct_key", "bank", "inst", "label", "split"
]
required_edge_arrays = [
    "src", "dst", "amount", "ts", "fmt", "is_laundering", "fmt_names"
]

print("Missing node columns:", sorted(set(required_node_cols) - set(nodes.columns)))
print("Missing edge arrays:", sorted(set(required_edge_arrays) - set(edges)))
print("Unique node_idx:", nodes["node_idx"].nunique())
print("Unique acct_key:", nodes["acct_key"].nunique())
print("Institutions:", sorted(nodes["inst"].unique().tolist()))
print("Splits:", sorted(nodes["split"].unique().tolist()))
print("Labels:", sorted(nodes["label"].unique().tolist()))

print("\nSplit counts:")
print(pd.crosstab(nodes["split"], nodes["label"]).to_string())

print("\nEdge index bounds:")
print("src min/max:", edges["src"].min(), edges["src"].max())
print("dst min/max:", edges["dst"].min(), edges["dst"].max())
print("Valid src:", bool(((edges["src"] >= 0) & (edges["src"] < len(nodes))).all()))
print("Valid dst:", bool(((edges["dst"] >= 0) & (edges["dst"] < len(nodes))).all()))

print("\nFeature caches:")
for view in ["combined", "B1", "B2", "B3", "B4", "B5"]:
    path = base / f"x_{view}.npy"
    x = np.load(path, mmap_mode="r")
    print(
        view,
        "shape:", x.shape,
        "finite:", bool(np.isfinite(x).all()),
        "exists:", path.exists()
    )

Missing node columns: []
Missing edge arrays: []
Unique node_idx: 515088
Unique acct_key: 515088
Institutions: ['B1', 'B2', 'B3', 'B4', 'B5']
Splits: ['test', 'train', 'val']
Labels: [0, 1]

Split counts:
label       0     1
split              
test   101747  1272
train  305238  3814
val    101746  1271

Edge index bounds:
src min/max: 1 515087
dst min/max: 0 515087
Valid src: True
Valid dst: True

Feature caches:
combined shape: (515088, 19) finite: True exists: True
B1 shape: (515088, 19) finite: True exists: True
B2 shape: (515088, 19) finite: True exists: True
B3 shape: (515088, 19) finite: True exists: True
B4 shape: (515088, 19) finite: True exists: True
B5 shape: (515088, 19) finite: True exists: True


In [24]:
import subprocess
from pathlib import Path

repo = Path("/home/ibm-user/shared")
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    capture_output=True,
    text=True
)
print(result.stdout or "(working tree clean)")
if result.stderr:
    print(result.stderr)
    

 M .gitignore
?? mulenet/
?? notebooks/
?? results/



In [25]:
import subprocess
from pathlib import Path

repo = Path("/home/ibm-user/shared")

for folder in ["mulenet", "notebooks", "results"]:
    print(f"\n--- {folder}/ ---")
    path = repo / folder
    if path.exists():
        for f in sorted(path.rglob("*")):
            if f.is_file():
                print(f"{f.relative_to(repo)} ({f.stat().st_size:,} bytes)")
    else:
        print("Folder not found")

print("\n--- .gitignore changes ---")
r = subprocess.run(
    ["git", "diff", "--", ".gitignore"],
    cwd=repo, capture_output=True, text=True
)
print(r.stdout or "(no tracked changes shown)")


--- mulenet/ ---
mulenet/.ipynb_checkpoints/enclave-checkpoint.py (3,840 bytes)
mulenet/.ipynb_checkpoints/enclave.py-checkpoint.preprivacy (3,132 bytes)
mulenet/.ipynb_checkpoints/enclave_v1_vipul-checkpoint.py (3,840 bytes)
mulenet/.ipynb_checkpoints/privacy_adapter_v1-checkpoint.py (1,544 bytes)
mulenet/__init__.py (0 bytes)
mulenet/__pycache__/__init__.cpython-312.pyc (138 bytes)
mulenet/__pycache__/build.cpython-312.pyc (9,962 bytes)
mulenet/__pycache__/common.cpython-312.pyc (14,808 bytes)
mulenet/__pycache__/enclave.cpython-312.pyc (5,034 bytes)
mulenet/__pycache__/gnn.cpython-312.pyc (19,987 bytes)
mulenet/__pycache__/privacy_adapter.cpython-312.pyc (2,310 bytes)
mulenet/__pycache__/toy.cpython-312.pyc (11,989 bytes)
mulenet/baselines.py (2,423 bytes)
mulenet/build.py (5,840 bytes)
mulenet/common.py (8,443 bytes)
mulenet/enclave.py (9,616 bytes)
mulenet/enclave.py.preprivacy (3,132 bytes)
mulenet/enclave_v1_vipul.py (3,840 bytes)
mulenet/evaluate.py (6,351 bytes)
mulenet/gnn.p